# 2AFC Constant-Stimuli Psychophysics Analysis

This notebook analyzes a two-alternative forced-choice (2AFC) constant-stimuli experiment, following the relevant perception-analysis structure used by Farajian & Nisky: count matrices of comparison-standard stiffness differences, logistic psychometric curves, PSE, and JND.

**Raw data rule:** set `DATA_ROOT` to the main data folder. The notebook treats it as read-only and writes all outputs to `OUTPUT_ROOT`.

**Important fitting rule:** the raw `answer` column is **not** used directly for fitting. Each trial is first converted into `response_comparison_greater` after determining whether the comparison stimulus was object 1 or object 2.

**Added for this experiment:** success/correctness is analyzed separately from the psychometric response, including answer duration (`time_to_answer`) and fatigue/learning proxies (trial order, elapsed time, first-vs-second half, and early/middle/late bins) per participant and across participants.


## 0. Configuration

Edit `DATA_SELECTION` to choose the cohort/subject to analyze:

- Group flags: `L_E`, `L_P`, `N_E`, `N_P`, `L_N_E`
- One subject: e.g. `L_E_14`

The notebook reads canonical subject folders from the new raw results folder and ignores folders marked `not finish` or `not include`. Outputs are written under `RUN_OUTPUT_ROOT` with separate `figures/` and `csv/` trees.

In [ ]:
from pathlib import Path
import sys
import json
import shutil
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Works whether Jupyter's current directory is the project root, analysis/, or this folder.
CWD = Path.cwd()
if (CWD / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD
elif (CWD / "analysis" / "psychophysics" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "analysis" / "psychophysics"
elif (CWD / "psychophysics" / "twoafc_psychophysics.py").exists():
    ANALYSIS_DIR = CWD / "psychophysics"
else:
    raise FileNotFoundError("Could not locate twoafc_psychophysics.py. Open the notebook from the project root, analysis/, or analysis/psychophysics/.")
sys.path.insert(0, str(ANALYSIS_DIR))
import twoafc_psychophysics as pf
pf = importlib.reload(pf)  # keep notebook kernel synced with local code edits

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# New read-only results folder. Expected subject folders: L_E_i, L_P_i, N_E_i, N_P_i.
DATA_ROOT = Path(r"C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\results")

# Choose one group, one subject, or a custom subject list.
# Examples: "L_E", "L_P", "N_E", "N_P", "L_N_E", "L_E_14", ["L_E_1", "L_E_2", "L_E_3"].
DATA_SELECTION = "N_E_1"
SELECTION_LABEL = pf.selection_label(DATA_SELECTION)

# Run-specific output folder. It is reset on each run so old outputs do not mix with the current selection.
OUTPUT_BASE = ANALYSIS_DIR / "results"
RUN_OUTPUT_ROOT = OUTPUT_BASE / SELECTION_LABEL
OVERWRITE_SELECTION_OUTPUT = True
KEEP_WORKING_OUTPUTS = False
if OVERWRITE_SELECTION_OUTPUT:
    pf.reset_output_root(RUN_OUTPUT_ROOT)
else:
    RUN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Intermediate notebook tables/figures are written here, then the requested tree is written to RUN_OUTPUT_ROOT.
OUTPUT_ROOT = RUN_OUTPUT_ROOT / "_working"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STANDARD_FALLBACK = 85.0          # used only as fallback/tie-break anchor
STANDARD_ABS_TOLERANCE = 0.75     # tolerance for classifying the inferred standard
FIG_DPI = 160

# Optional manual column overrides if automatic detection needs help.
# Example: {"object_1_value": "object_1_stiffness", "answer": "answer"}
MANUAL_COLUMN_OVERRIDES = {}

print("ANALYSIS_DIR:", ANALYSIS_DIR.resolve())
print("DATA_ROOT:", DATA_ROOT)
print("DATA_SELECTION:", DATA_SELECTION, "->", SELECTION_LABEL)
print("RUN_OUTPUT_ROOT:", RUN_OUTPUT_ROOT.resolve())
print("WORKING OUTPUT_ROOT:", OUTPUT_ROOT.resolve())

ANALYSIS_DIR: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics
DATA_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\results
DATA_SELECTION: ('N_E', 'L_E') -> custom_N_E_plus_L_E
RUN_OUTPUT_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics\results\custom_N_E_plus_L_E
WORKING OUTPUT_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics\results\custom_N_E_plus_L_E\_working


## 1. File discovery and loading

One canonical folder per subject is expected under `DATA_ROOT` (`L_E_i`, `L_P_i`, `N_E_i`, `N_P_i`). Folder names containing `not finish` or `not include` are skipped. The code searches each included subject folder recursively for files named exactly `answers.csv`, then saves a discovery audit table.

In [258]:
pf.validate_paths(DATA_ROOT, RUN_OUTPUT_ROOT)

file_discovery_summary = pf.discover_answer_files(DATA_ROOT, RUN_OUTPUT_ROOT, selection=DATA_SELECTION)

# Subject folders are siblings of the group folder, e.g. results/L_E and results/L_E_1, results/L_E_2, ...
# Reset only the subjects included in this run so old subject outputs do not mix with the current run.
selected_subject_ids = sorted(
    file_discovery_summary.loc[file_discovery_summary["selected"], "canonical_subject_id"].dropna().astype(str).unique(),
    key=pf._subject_sort_key,
)
if OVERWRITE_SELECTION_OUTPUT:
    for subject_id in selected_subject_ids:
        pf.reset_output_root(OUTPUT_BASE / pf.sanitize_name(subject_id))

pf.save_csv(file_discovery_summary, OUTPUT_ROOT, "file_discovery_summary.csv")
display(file_discovery_summary.head(30))

combined_raw_imported_data = pf.load_selected_subject_csvs(file_discovery_summary)
pf.save_csv(combined_raw_imported_data, OUTPUT_ROOT, "combined_raw_imported_data.csv")
display(combined_raw_imported_data.head())
print("combined raw shape:", combined_raw_imported_data.shape)
print("included subjects:", sorted(combined_raw_imported_data["subject_id"].astype(str).unique(), key=pf._subject_sort_key))
print("group output folder:", RUN_OUTPUT_ROOT)
print("subject output folders:", [OUTPUT_BASE / pf.sanitize_name(s) for s in selected_subject_ids])

,subject_id,canonical_subject_id,subject_group_code,subject_folder,candidate_rank,candidate_score,selected,source_file,source_file_name,n_answer_csv_candidates,n_total_csv_files_in_subject_folder,selection_warning
0,L_E_1,L_E_1,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
1,L_E_10,L_E_10,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,272,
2,L_E_11,L_E_11,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
3,L_E_12,L_E_12,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
4,L_E_14,L_E_14,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
5,L_E_15,L_E_15,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
6,L_E_16,L_E_16,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
7,L_E_17,L_E_17,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
8,L_E_18,L_E_18,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,271,
9,L_E_19,L_E_19,L_E,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1,37,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,272,


,subject_id,_subject_folder,_source_file,_source_file_name,_row_in_source,timestamp,pair_number,object_1_finger,object_1_stiffness,object_2_finger,object_2_stiffness,time_to_answer,answer
0,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,0,2026-05-11T11:02:36.482977,1,index,85,index,40,5.399998,0
1,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,2026-05-11T11:03:01.054278,2,index,85,index,130,2.521047,1
2,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,2,2026-05-11T11:03:19.190788,3,index,70,index,85,2.313266,1
3,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,3,2026-05-11T11:03:57.492458,4,middle,85,middle,70,2.368027,0
4,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,4,2026-05-11T11:04:11.951749,5,middle,130,middle,85,2.532164,0


combined raw shape: (9648, 13)
included subjects: ['L_E_1', 'L_E_10', 'L_E_11', 'L_E_12', 'L_E_14', 'L_E_15', 'L_E_16', 'L_E_17', 'L_E_18', 'L_E_19', 'L_E_2', 'L_E_3', 'L_E_4', 'L_E_5', 'L_E_6', 'L_E_7', 'L_E_8', 'L_E_9', 'N_E_1', 'N_E_11', 'N_E_12', 'N_E_13', 'N_E_14', 'N_E_15', 'N_E_16', 'N_E_17', 'N_E_18', 'N_E_19', 'N_E_2', 'N_E_3', 'N_E_4', 'N_E_5', 'N_E_6', 'N_E_7', 'N_E_8', 'N_E_9']
group output folder: c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\psychophysics\results\custom_N_E_plus_L_E
subject output folders: [WindowsPath('c:/Users/user/BIO MEDICAL ROBOTICS Dropbox/Elisheva Shiri Decktor/BGU/Codes/Parallel_Heptics/analysis/psychophysics/results/L_E_1'), WindowsPath('c:/Users/user/BIO MEDICAL ROBOTICS Dropbox/Elisheva Shiri Decktor/BGU/Codes/Parallel_Heptics/analysis/psychophysics/results/L_E_10'), WindowsPath('c:/Users/user/BIO MEDICAL ROBOTICS Dropbox/Elisheva Shiri Decktor/BGU/Codes/Parallel_Heptics/analysis/psychophy

## 2. Column detection and validation

Detects object values, object fingers, answer, reaction time, trial index, timestamp, and optional block columns. Review the detection table; if needed, set `MANUAL_COLUMN_OVERRIDES` above and rerun.

In [259]:
detected_columns, column_detection_details = pf.detect_columns(combined_raw_imported_data, MANUAL_COLUMN_OVERRIDES)
print(json.dumps(detected_columns, indent=2, ensure_ascii=False))
pf.save_csv(column_detection_details, OUTPUT_ROOT, "column_detection_details.csv")
display(column_detection_details.groupby("target").head(5))

missing_required = [k for k in ["object_1_value", "object_2_value", "answer"] if not detected_columns.get(k)]
if missing_required:
    raise RuntimeError(f"Missing required detected columns: {missing_required}. Add MANUAL_COLUMN_OVERRIDES and rerun.")

{
  "object_1_value": "object_1_stiffness",
  "object_2_value": "object_2_stiffness",
  "object_1_finger": "object_1_finger",
  "object_2_finger": "object_2_finger",
  "answer": "answer",
  "reaction_time": "time_to_answer",
  "trial_index": "pair_number",
  "timestamp": "timestamp",
  "block": null
}


,target,column,normalized,score,manual
0,object_1_value,object_1_stiffness,object_1_stiffness,23,False
1,object_1_value,object_2_stiffness,object_2_stiffness,9,False
2,object_1_value,object_1_finger,object_1_finger,6,False
3,object_1_value,_row_in_source,row_in_source,0,False
4,object_1_value,_source_file,source_file,0,False
8,object_2_value,object_2_stiffness,object_2_stiffness,23,False
9,object_2_value,object_1_stiffness,object_1_stiffness,9,False
10,object_2_value,object_2_finger,object_2_finger,6,False
11,object_2_value,_row_in_source,row_in_source,0,False
12,object_2_value,_source_file,source_file,0,False


## 3. Trial cleaning and canonicalization

Protocol rows and invalid/ambiguous trials are flagged and saved separately. Valid trials receive `response_comparison_greater` for fitting.

Examples:
- standard object 1, comparison object 2 ? response is 1 when `answer == 1`.
- standard object 2, comparison object 1 ? response is 1 when `answer == 0`.

In [260]:
inferred_standard_value, standard_inference_table = pf.infer_standard_value(combined_raw_imported_data, detected_columns, STANDARD_FALLBACK)
print(f"Inferred standard/reference value: {inferred_standard_value:g}")
pf.save_csv(standard_inference_table, OUTPUT_ROOT, "standard_value_inference.csv")
display(standard_inference_table.head(10))

combined_clean_trials, combined_flagged_trials = pf.canonicalize_trials(
    combined_raw_imported_data,
    detected_columns,
    standard_value=inferred_standard_value,
    standard_tolerance=STANDARD_ABS_TOLERANCE,
)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(combined_flagged_trials, OUTPUT_ROOT, "combined_flagged_trials.csv")

print("Clean trials:", combined_clean_trials.shape)
display(combined_clean_trials.head())
print("Flagged trials:", combined_flagged_trials.shape)
display(combined_flagged_trials.head())

Inferred standard/reference value: 85


,candidate_value,count,n_unique_partners,distance_from_fallback,standard_score
0,85,9648,8,0.0,1.350000
1,70,1296,1,15.0,0.165578
2,100,1152,1,15.0,0.150653
3,130,1296,1,45.0,0.140578
4,40,1296,1,45.0,0.140578
5,115,1152,1,30.0,0.138153
6,55,1152,1,30.0,0.138153
7,145,1152,1,60.0,0.113153
8,25,1152,1,60.0,0.113153


Clean trials: (9216, 30)


,subject_id,source_file,source_file_name,row_in_source,trial_index_raw,timestamp,block_raw,object_1_value,object_2_value,object_1_finger_raw,object_2_finger_raw,object_1_finger,object_2_finger,standard_value,standard_side,comparison_side,comparison_value,standard_finger,comparison_finger,finger_condition,raw_answer,answer_code,response_comparison_greater,reaction_time,protocol_marker,finger_warning,excluded_from_fit,flag_reason,global_trial_order,answer_chose_object_2
0,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,12,13,2026-05-11T11:07:26.541520,NaN,115,85,ring,ring,R,R,85.0,object_2,object_1,115.0,R,R,R,0,0,1,2.323103,False,,False,,13,0.0
1,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,13,14,2026-05-11T11:07:46.816911,NaN,55,85,ring,ring,R,R,85.0,object_2,object_1,55.0,R,R,R,1,1,0,2.351829,False,,False,,14,1.0
2,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,14,15,2026-05-11T11:08:03.041546,NaN,40,85,ring,ring,R,R,85.0,object_2,object_1,40.0,R,R,R,0,0,1,2.152518,False,,False,,15,0.0
3,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,15,16,2026-05-11T11:08:29.746061,NaN,145,85,ring,ring,R,R,85.0,object_2,object_1,145.0,R,R,R,0,0,1,2.168074,False,,False,,16,0.0
4,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,16,17,2026-05-11T11:08:46.876595,NaN,85,130,ring,ring,R,R,85.0,object_1,object_2,130.0,R,R,R,1,1,1,2.160077,False,,False,,17,1.0


Flagged trials: (432, 30)


,subject_id,source_file,source_file_name,row_in_source,trial_index_raw,timestamp,block_raw,object_1_value,object_2_value,object_1_finger_raw,object_2_finger_raw,object_1_finger,object_2_finger,standard_value,standard_side,comparison_side,comparison_value,standard_finger,comparison_finger,finger_condition,raw_answer,answer_code,response_comparison_greater,reaction_time,protocol_marker,finger_warning,excluded_from_fit,flag_reason,global_trial_order,answer_chose_object_2
0,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,0,1,2026-05-11T11:02:36.482977,NaN,85,40,index,index,I,I,85.0,object_1,object_2,40.0,I,I,I,0,0,0,5.399998,False,,True,pilot_onboarding_excluded,1,0.0
1,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,2,2026-05-11T11:03:01.054278,NaN,85,130,index,index,I,I,85.0,object_1,object_2,130.0,I,I,I,1,1,1,2.521047,False,,True,pilot_onboarding_excluded,2,1.0
2,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,2,3,2026-05-11T11:03:19.190788,NaN,70,85,index,index,I,I,85.0,object_2,object_1,70.0,I,I,I,1,1,0,2.313266,False,,True,pilot_onboarding_excluded,3,1.0
3,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,3,4,2026-05-11T11:03:57.492458,NaN,85,70,middle,middle,M,M,85.0,object_1,object_2,70.0,M,M,M,0,0,0,2.368027,False,,True,pilot_onboarding_excluded,4,0.0
4,L_E_1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,4,5,2026-05-11T11:04:11.951749,NaN,130,85,middle,middle,M,M,85.0,object_2,object_1,130.0,M,M,M,0,0,1,2.532164,False,,True,pilot_onboarding_excluded,5,0.0


## 4. Block and finger detection

Finger labels are normalized to `I`, `M`, `R`, `P`. Block order is inferred from changes in `finger_condition` within each subject.

In [261]:
combined_clean_trials, block_order_summary = pf.add_block_numbers(combined_clean_trials)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(block_order_summary, OUTPUT_ROOT, "block_order_summary.csv")
display(block_order_summary.head(30))
display(pd.crosstab(combined_clean_trials["subject_id"], combined_clean_trials["finger_condition"]))

,subject_id,block_number,finger_condition,first_global_trial_order,n_trials_in_block
0,L_E_1,1,R,13,64
1,L_E_1,2,P,77,64
2,L_E_1,3,M,141,64
3,L_E_1,4,I,205,64
4,L_E_10,1,I,13,64
5,L_E_10,2,M,77,64
6,L_E_10,3,P,141,64
7,L_E_10,4,R,205,64
8,L_E_11,1,P,13,64
9,L_E_11,2,I,77,64


finger_condition,I,M,P,R
subject_id,,,,
L_E_1,64,64,64,64
L_E_10,64,64,64,64
L_E_11,64,64,64,64
L_E_12,64,64,64,64
L_E_14,64,64,64,64
L_E_15,64,64,64,64
L_E_16,64,64,64,64
L_E_17,64,64,64,64
L_E_18,64,64,64,64


## 5. Success/correctness coding and Farajian-style count matrices

Psychometric fitting uses `response_comparison_greater`. Success-rate analysis asks a different question: did the participant choose the physically stiffer stimulus? This cell adds `correct_response`, signed stiffness deltas, elapsed time, participant group (`pilot`/`experiment`), and fatigue/order proxies.


In [262]:
combined_success_trials = pf.add_success_and_time_columns(combined_clean_trials)
pf.save_csv(combined_success_trials, OUTPUT_ROOT, "combined_success_trials.csv")

farajian_style_input_by_subject_finger = pf.make_farajian_style_psychometric_input(combined_success_trials, ["subject_id", "finger_condition"])
farajian_style_input_group_by_finger = pf.make_farajian_style_psychometric_input(combined_success_trials, ["finger_condition"])
pf.save_csv(farajian_style_input_by_subject_finger, OUTPUT_ROOT, "farajian_style_input_by_subject_finger.csv")
pf.save_csv(farajian_style_input_group_by_finger, OUTPUT_ROOT, "farajian_style_input_group_by_finger.csv")

display(combined_success_trials[[
    "subject_id", "subject_group_label", "finger_condition", "global_trial_order",
    "signed_stiffness_delta", "response_comparison_greater", "correct_response",
    "reaction_time", "elapsed_minutes", "session_half", "fatigue_tertile"
]].head(20))
display(farajian_style_input_by_subject_finger.head(30))


,subject_id,subject_group_label,finger_condition,global_trial_order,signed_stiffness_delta,response_comparison_greater,correct_response,reaction_time,elapsed_minutes,session_half,fatigue_tertile
0,L_E_1,other,R,13,30.0,1,1.0,2.323103,0.000000,first_half,early
1,L_E_1,other,R,14,-30.0,0,1.0,2.351829,0.337923,first_half,early
2,L_E_1,other,R,15,-45.0,1,0.0,2.152518,0.608334,first_half,early
3,L_E_1,other,R,16,60.0,1,1.0,2.168074,1.053409,first_half,early
4,L_E_1,other,R,17,45.0,1,1.0,2.160077,1.338918,first_half,early
5,L_E_1,other,R,18,-15.0,1,0.0,2.202255,1.625668,first_half,early
6,L_E_1,other,R,19,15.0,1,1.0,2.254923,1.862973,first_half,early
7,L_E_1,other,R,20,-60.0,0,1.0,2.053244,2.103824,first_half,early
8,L_E_1,other,R,21,-15.0,0,1.0,2.129709,2.350896,first_half,early
9,L_E_1,other,R,22,-30.0,0,1.0,2.054690,2.597496,first_half,early


,subject_id,finger_condition,delta_comparison_minus_standard,delta_standard_minus_comparison,n_comparison_greater,n_comparison_less,n_trials,n_data_points,p_comparison_greater,p_comparison_greater_ci95_lower,p_comparison_greater_ci95_upper,p_comparison_less,p_comparison_less_ci95_lower,p_comparison_less_ci95_upper,n_correct,success_rate,success_rate_ci95_lower,success_rate_ci95_upper,mean_reaction_time,median_reaction_time,reaction_time_ci95_lower,reaction_time_ci95_upper,mean_log_reaction_time,geomean_reaction_time_s,reaction_time_log_ci95_lower_backtransformed_s,reaction_time_log_ci95_upper_backtransformed_s,standard_value,comparison_value,comparison_over_standard,signed_delta_over_standard,abs_delta_over_standard,signed_stiffness_delta,log_reaction_time_ci95_lower,log_reaction_time_ci95_upper
0,L_E_1,I,-60.0,60.0,0,8,8,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,8.0,1.000,0.675584,1.000000,1.754411,1.747458,1.722376,1.786447,0.561830,1.753879,1.722173,1.786170,85.0,25.0,0.294118,-0.705882,0.705882,-60.0,0.543587,0.580073
1,L_E_1,I,-45.0,45.0,0,8,8,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,8.0,1.000,0.675584,1.000000,1.843788,1.798201,1.747599,1.939977,0.609462,1.839441,1.749236,1.934297,85.0,40.0,0.470588,-0.529412,0.529412,-45.0,0.559179,0.659744
2,L_E_1,I,-30.0,30.0,0,8,8,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,8.0,1.000,0.675584,1.000000,1.748191,1.720479,1.697054,1.799328,0.557816,1.746853,1.697175,1.797986,85.0,55.0,0.647059,-0.352941,0.352941,-30.0,0.528965,0.586667
3,L_E_1,I,-15.0,15.0,2,6,8,8,0.250,0.071478,0.590730,0.750,0.409270,0.928522,6.0,0.750,0.409270,0.928522,1.790395,1.769528,1.733259,1.847531,0.581516,1.788748,1.732884,1.846412,85.0,70.0,0.823529,-0.176471,0.176471,-15.0,0.549787,0.613244
4,L_E_1,I,15.0,-15.0,6,2,8,8,0.750,0.409270,0.928522,0.250,0.071478,0.590730,6.0,0.750,0.409270,0.928522,1.837103,1.776203,1.728583,1.945624,0.605233,1.831679,1.732002,1.937092,85.0,100.0,1.176471,0.176471,0.176471,15.0,0.549278,0.661188
5,L_E_1,I,30.0,-30.0,6,2,8,8,0.750,0.409270,0.928522,0.250,0.071478,0.590730,6.0,0.750,0.409270,0.928522,1.757840,1.762569,1.705581,1.810098,0.563284,1.756431,1.705155,1.809250,85.0,115.0,1.352941,0.352941,0.352941,30.0,0.533656,0.592912
6,L_E_1,I,45.0,-45.0,7,1,8,8,0.875,0.529105,0.977583,0.125,0.022417,0.470895,7.0,0.875,0.529105,0.977583,1.789623,1.776900,1.745184,1.834063,0.581452,1.788634,1.745283,1.833062,85.0,130.0,1.529412,0.529412,0.529412,45.0,0.556917,0.605988
7,L_E_1,I,60.0,-60.0,8,0,8,8,1.000,0.675584,1.000000,0.000,0.000000,0.324416,8.0,1.000,0.675584,1.000000,1.818003,1.797513,1.724315,1.911691,0.595410,1.813775,1.725160,1.906941,85.0,145.0,1.705882,0.705882,0.705882,60.0,0.545320,0.645501
8,L_E_1,M,-60.0,60.0,0,8,8,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,8.0,1.000,0.675584,1.000000,1.912087,1.806236,1.687564,2.136610,0.637865,1.892436,1.710083,2.094235,85.0,25.0,0.294118,-0.705882,0.705882,-60.0,0.536542,0.739189
9,L_E_1,M,-45.0,45.0,0,8,8,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,8.0,1.000,0.675584,1.000000,1.795579,1.786498,1.766832,1.824326,0.585095,1.795162,1.766780,1.824000,85.0,40.0,0.470588,-0.529412,0.529412,-45.0,0.569158,0.601032


## 6. Quality-control summaries

Flags suspicious cases without silently deleting more data: missing/invalid rows, trial counts per value/finger, non-monotonic curves, apparent error rate, RT outliers, side bias, standard-side bias, and related warnings.

In [263]:
qc_summary = pf.make_qc_summary(combined_clean_trials, combined_flagged_trials)
pf.save_csv(qc_summary, OUTPUT_ROOT, "qc_summary.csv")
display(qc_summary)

,subject_id,finger_condition,n_clean_trials,n_stimulus_levels,min_trials_per_value,median_trials_per_value,side_bias_p_chose_object2,standard_side_p_object2,apparent_error_rate,rt_outlier_count,monotonic_violations,spearman_r,non_monotonic_flag,qc_warnings,n_flagged_rows
0,L_E_1,I,64,8,8,8.0,0.515625,0.5,0.109375,1,0,0.969782,False,,12
1,L_E_1,M,64,8,8,8.0,0.390625,0.5,0.140625,1,0,0.981981,False,,12
2,L_E_1,P,64,8,8,8.0,0.375000,0.5,0.218750,2,0,0.897118,False,,12
3,L_E_1,R,64,8,8,8.0,0.500000,0.5,0.281250,3,2,0.690476,True,non_monotonic_curve,12
4,L_E_10,I,64,8,8,8.0,0.515625,0.5,0.234375,4,0,0.945611,False,,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,N_E_8,R,64,8,8,8.0,0.406250,0.5,0.156250,2,1,0.891631,True,non_monotonic_curve,12
140,N_E_9,I,64,8,8,8.0,0.515625,0.5,0.171875,3,0,0.906687,False,,12
141,N_E_9,M,64,8,8,8.0,0.531250,0.5,0.187500,1,1,0.927778,True,non_monotonic_curve,12
142,N_E_9,P,64,8,8,8.0,0.484375,0.5,0.234375,3,0,0.988024,False,,12


## 7. Psychometric aggregation

Aggregates valid trials into binomial counts: `comparison_value`, `n_trials`, `n_comparison_greater`, and observed probability.

In [264]:
# Enrich-once: feed the already-enriched `combined_success_trials` into
# make_psychometric_input (it carries log_reaction_time, so the helper uses its
# fast pre-enriched branch instead of re-running add_success_and_time_columns).
psychometric_input_by_subject_finger = pf.make_psychometric_input(combined_success_trials, ["subject_id", "finger_condition"])
pf.save_csv(psychometric_input_by_subject_finger, OUTPUT_ROOT, "psychometric_input_by_subject_finger.csv")
display(psychometric_input_by_subject_finger.head(30))

,subject_id,finger_condition,comparison_value,n_trials,n_data_points,n_comparison_greater,n_comparison_less,p_comparison_greater,p_comparison_greater_ci95_lower,p_comparison_greater_ci95_upper,p_comparison_less,p_comparison_less_ci95_lower,p_comparison_less_ci95_upper,mean_rt,median_rt,rt_ci95_lower,rt_ci95_upper,mean_log_reaction_time,log_reaction_time_ci95_lower,log_reaction_time_ci95_upper,standard_value,comparison_over_standard,signed_delta_over_standard,abs_delta_over_standard,geomean_reaction_time_s,reaction_time_log_ci95_lower_backtransformed_s,reaction_time_log_ci95_upper_backtransformed_s,delta_comparison_minus_standard,delta_standard_minus_comparison
0,L_E_1,I,25.0,8,8,0,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,1.754411,1.747458,1.722376,1.786447,0.561830,0.543587,0.580073,85.0,0.294118,-0.705882,0.705882,1.753879,1.722173,1.786170,-60.0,60.0
1,L_E_1,I,40.0,8,8,0,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,1.843788,1.798201,1.747599,1.939977,0.609462,0.559179,0.659744,85.0,0.470588,-0.529412,0.529412,1.839441,1.749236,1.934297,-45.0,45.0
2,L_E_1,I,55.0,8,8,0,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,1.748191,1.720479,1.697054,1.799328,0.557816,0.528965,0.586667,85.0,0.647059,-0.352941,0.352941,1.746853,1.697175,1.797986,-30.0,30.0
3,L_E_1,I,70.0,8,8,2,6,0.250,0.071478,0.590730,0.750,0.409270,0.928522,1.790395,1.769528,1.733259,1.847531,0.581516,0.549787,0.613244,85.0,0.823529,-0.176471,0.176471,1.788748,1.732884,1.846412,-15.0,15.0
4,L_E_1,I,100.0,8,8,6,2,0.750,0.409270,0.928522,0.250,0.071478,0.590730,1.837103,1.776203,1.728583,1.945624,0.605233,0.549278,0.661188,85.0,1.176471,0.176471,0.176471,1.831679,1.732002,1.937092,15.0,-15.0
5,L_E_1,I,115.0,8,8,6,2,0.750,0.409270,0.928522,0.250,0.071478,0.590730,1.757840,1.762569,1.705581,1.810098,0.563284,0.533656,0.592912,85.0,1.352941,0.352941,0.352941,1.756431,1.705155,1.809250,30.0,-30.0
6,L_E_1,I,130.0,8,8,7,1,0.875,0.529105,0.977583,0.125,0.022417,0.470895,1.789623,1.776900,1.745184,1.834063,0.581452,0.556917,0.605988,85.0,1.529412,0.529412,0.529412,1.788634,1.745283,1.833062,45.0,-45.0
7,L_E_1,I,145.0,8,8,8,0,1.000,0.675584,1.000000,0.000,0.000000,0.324416,1.818003,1.797513,1.724315,1.911691,0.595410,0.545320,0.645501,85.0,1.705882,0.705882,0.705882,1.813775,1.725160,1.906941,60.0,-60.0
8,L_E_1,M,25.0,8,8,0,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,1.912087,1.806236,1.687564,2.136610,0.637865,0.536542,0.739189,85.0,0.294118,-0.705882,0.705882,1.892436,1.710083,2.094235,-60.0,60.0
9,L_E_1,M,40.0,8,8,0,8,0.000,0.000000,0.324416,1.000,0.675584,1.000000,1.795579,1.786498,1.766832,1.824326,0.585095,0.569158,0.601032,85.0,0.470588,-0.529412,0.529412,1.795162,1.766780,1.824000,-45.0,45.0


## Psychometric fitting setup

The notebook attempts Wichmann-lab `psignifit` first when available and compatible. If not, it prints installation guidance and uses the scipy logistic fallback on physical stimulus values.

## Psychometric model: lapse-aware yes/no fit

Each subject-by-finger psychometric function is fitted to **P(y=1)** -- the probability that the participant judged the **comparison** stimulus C as stiffer than the fixed **standard** S = 85 (response in `response_comparison_greater`; this is the participant's perceptual judgement, **not** percent-correct, side, order, or physical truth).

Because this is a yes/no probability (not a 2AFC percent-correct curve with a fixed guess rate), we fit a **lapse-aware sigmoid with four free parameters** (`mu`, `scale`, `lapse_low`, `lapse_high`):

$$P(y=1) = \lambda_	ext{low} + (1 - \lambda_	ext{low} - \lambda_	ext{high})\,F(\delta;\ \mu,\ 	ext{scale}),\qquad \delta = C - 85$$

where $F$ is a monotonic logistic sigmoid.

**Fitter used:** `psignifit` when installed (preferred); otherwise the custom lapse-aware maximum-likelihood fitter (`fit_with_scipy_logistic`). The fitter for each fit is recorded in the `fit_method` / `psignifit_status` columns.

**Derived quantities** (the fit is in comparison-stiffness units, so these equal the delta-space definitions):
- **PSE** = comparison value where P(y=1)=0.5; **Bias** = PSE - 85 = delta at P(y=1)=0.5 (`pse_delta_from_standard`).
- **JND** = (x75 - x25)/2, where x25/x75 are where the curve reaches 0.25/0.75. Because of the lapse asymptotes, 0.25/0.75 are only attainable within `[lapse_low, 1 - lapse_high]`; when they are not, PSE/JND are returned as NaN and flagged in `fit_warning` (`pse_outside_lapse_range` / `jnd_quantile_outside_lapse_range`) rather than reported as invalid.


In [265]:
PSIGNIFIT_AVAILABLE, PSIGNIFIT_STATUS = pf.check_psignifit_available()
print(PSIGNIFIT_STATUS)
if not PSIGNIFIT_AVAILABLE:
    print("Optional psignifit install example: pip install psignifit")
    print("Continuing with scipy logistic fallback fits.")

psignifit import succeeded (version=unknown)


## 8. Subject x finger psychometric fits

In [266]:
# Fit per subject/finger first (ALL subjects, incl. pilots — kept for the
# individual curves). The GROUP-level fits below then pool ONLY valid subjects'
# trials, so pilot/protocol subjects (*_P) and band-pass-disqualified subjects
# (PSE outside 25-145) do not contaminate the experiment-group result.
pse_jnd_by_subject_finger = pf.fit_conditions(
    psychometric_input_by_subject_finger, ["subject_id", "finger_condition"], PSIGNIFIT_AVAILABLE
)
pf.save_csv(pse_jnd_by_subject_finger, OUTPUT_ROOT, "pse_jnd_by_subject_finger.csv")

# Valid subject/finger fits for the selected all/group analysis. The selection has already
# removed other cohorts/subjects, so keep both experiment and protocol subjects here.
# Only the PSE reliability band excludes a subject/finger from all-level pooling.
valid_group_fits = pse_jnd_by_subject_finger.copy()
if "excluded_from_group_analysis" in valid_group_fits.columns:
    valid_group_fits = valid_group_fits[~valid_group_fits["excluded_from_group_analysis"].astype(bool)]
_valid_keys = set(valid_group_fits["subject_id"].astype(str) + "||" + valid_group_fits["finger_condition"].astype(str))
_trial_key = combined_success_trials["subject_id"].astype(str) + "||" + combined_success_trials["finger_condition"].astype(str)
group_trials = combined_success_trials[_trial_key.isin(_valid_keys)].copy()
print(
    f"All/group analysis for {DATA_SELECTION} uses {group_trials['subject_id'].nunique()} selected subjects "
    f"({len(group_trials)} of {len(combined_success_trials)} trials after the PSE reliability band)."
)

psychometric_input_subject_pooled = pf.make_psychometric_input(group_trials, ["subject_id"])
psychometric_input_group_by_finger = pf.make_psychometric_input(group_trials, ["finger_condition"])
all_pooled_trials = group_trials.copy()
all_pooled_trials["group"] = "all_pooled"
psychometric_input_group_all_pooled = pf.make_psychometric_input(all_pooled_trials, ["group"])

pf.save_csv(psychometric_input_subject_pooled, OUTPUT_ROOT, "psychometric_input_subject_pooled.csv")
pf.save_csv(psychometric_input_group_by_finger, OUTPUT_ROOT, "psychometric_input_group_by_finger.csv")
pf.save_csv(psychometric_input_group_all_pooled, OUTPUT_ROOT, "psychometric_input_group_all_pooled.csv")

(
    pse_jnd_subject_pooled,
    pse_jnd_group_by_finger,
    pse_jnd_group_all_pooled,
) = pf.fit_conditions_many(
    [
        (psychometric_input_subject_pooled, ["subject_id"]),
        (psychometric_input_group_by_finger, ["finger_condition"]),
        (psychometric_input_group_all_pooled, ["group"]),
    ],
    PSIGNIFIT_AVAILABLE,
)

pf.save_csv(pse_jnd_subject_pooled, OUTPUT_ROOT, "pse_jnd_subject_pooled.csv")
pf.save_csv(pse_jnd_group_by_finger, OUTPUT_ROOT, "pse_jnd_group_by_finger.csv")
pf.save_csv(pse_jnd_group_all_pooled, OUTPUT_ROOT, "pse_jnd_group_all_pooled.csv")
display(pse_jnd_by_subject_finger)

All/group analysis for ('N_E', 'L_E') uses 35 selected subjects (8704 of 9216 trials after the PSE reliability band).


,subject_id,finger_condition,fit_method,pse,pse_se,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,standard_inside_pse_ci95,pse_bias_p_value,jnd,jnd_se,jnd_ci95_lower,jnd_ci95_upper,weber_fraction,jnd_over_standard,x25,x75,slope_at_pse,lapse_rate,lapse_rate_ci95_lower,lapse_rate_ci95_upper,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,n_bootstrap,bootstrap_method,deviance,aic,psignifit_status,neg_log_likelihood,lapse_rate_se,n_bootstrap_effective_pse,n_bootstrap_effective_jnd,mu,scale,standard_value,comparison_min,comparison_max,experiment_group,experiment_group_label,setup_factor,setup_label,protocol_group,protocol_group_label,success_label,workspace_setup,workspace_width_cm,workspace_height_cm,workspace_label,protocol_factor,sex_factor,age_years,age_group,pse_delta_comparison_minus_standard,pse_delta_standard_minus_comparison,abs_pse_delta_from_standard,pse_in_valid_band,excluded_from_group_analysis,group_exclusion_reason,pse_delta_se,jnd_over_standard_ci95_lower,jnd_over_standard_ci95_upper
0,L_E_1,I,scipy_logistic_fallback,91.108696,9.009330,70.408383,100.869088,6.108696,-14.591617,15.869088,1.0,0.497746,14.535883,5.846663,0.812416,18.324958,0.171010,0.171010,76.918570,105.990336,0.019068,0.035741,0.000000,0.200000,0.000000,0.035741,ok,,64,8,200,parametric_binomial,3.519438,44.539848,psignifit_failed: 'dict' object has no attribu...,18.269924,0.061682,200,200,90.172331,12.625223,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,6.108696,-6.108696,6.108696,True,False,,9.009330,0.009558,0.215588
1,L_E_1,M,scipy_logistic_fallback,82.438040,9.059173,66.808820,99.055071,-2.561960,-18.191180,14.055071,1.0,0.777328,16.756857,6.174225,0.774389,19.882109,0.197139,0.197139,65.681183,99.194897,0.016390,0.000000,0.000000,0.197970,0.000000,0.000000,ok,,64,8,200,parametric_binomial,4.930972,48.044374,psignifit_failed: 'dict' object has no attribu...,20.022187,0.057824,200,200,82.438040,15.252749,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,-2.561960,2.561960,2.561960,True,False,,9.059173,0.009110,0.233907
2,L_E_1,P,scipy_logistic_fallback,69.531903,5.344104,56.320824,84.286972,-15.468097,-28.679176,-0.713028,0.0,0.003799,1.086504,5.054087,0.845586,21.479602,0.012782,0.012782,68.695145,70.868153,0.302197,0.241667,0.062845,0.312763,0.041667,0.200000,warning,high_lapse_estimate,64,8,200,parametric_binomial,4.139365,63.360759,psignifit_failed: 'dict' object has no attribu...,27.680379,0.056546,200,200,69.277613,0.600002,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,-15.468097,15.468097,15.468097,True,False,,5.344104,0.009948,0.252701
3,L_E_1,R,scipy_logistic_fallback,68.767526,9.326705,54.993778,91.465298,-16.232474,-30.006222,6.465298,1.0,0.081784,19.096955,9.318958,1.048644,34.840626,0.224670,0.224670,54.574880,92.768789,0.017165,0.200000,0.000000,0.325469,0.000000,0.200000,warning,high_lapse_estimate,64,8,200,parametric_binomial,9.201238,79.512986,psignifit_failed: 'dict' object has no attribu...,35.756493,0.076443,200,200,63.187550,10.923445,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,-16.232474,16.232474,16.232474,True,False,,9.326705,0.012337,0.409890
4,L_E_10,I,scipy_logistic_fallback,90.499935,12.320023,69.479502,113.654648,5.499935,-15.520498,28.654648,1.0,0.655292,30.808008,13.547575,1.204485,44.997789,0.362447,0.362447,66.782550,128.398565,0.010657,0.241262,0.000000,0.386383,0.041262,0.200000,warning,high_lapse_estimate,64,8,200,parametric_binomial,2.699994,72.506399,psignifit_failed: 'dict' object has no attribu...,32

## 9. Subject-level pooled-across-fingers fits

In [267]:
# Fitted above in one shared parallel pool; display the subject-pooled result.
display(pse_jnd_subject_pooled)

,subject_id,fit_method,pse,pse_se,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,standard_inside_pse_ci95,pse_bias_p_value,jnd,jnd_se,jnd_ci95_lower,jnd_ci95_upper,weber_fraction,jnd_over_standard,x25,x75,slope_at_pse,lapse_rate,lapse_rate_ci95_lower,lapse_rate_ci95_upper,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,n_bootstrap,bootstrap_method,deviance,aic,psignifit_status,neg_log_likelihood,lapse_rate_se,n_bootstrap_effective_pse,n_bootstrap_effective_jnd,mu,scale,standard_value,comparison_min,comparison_max,experiment_group,experiment_group_label,setup_factor,setup_label,protocol_group,protocol_group_label,success_label,workspace_setup,workspace_width_cm,workspace_height_cm,workspace_label,protocol_factor,sex_factor,age_years,age_group,pse_delta_comparison_minus_standard,pse_delta_standard_minus_comparison,abs_pse_delta_from_standard,pse_in_valid_band,excluded_from_group_analysis,group_exclusion_reason,pse_delta_se,jnd_over_standard_ci95_lower,jnd_over_standard_ci95_upper
0,L_E_1,scipy_logistic_fallback,72.982062,3.982017,66.470491,81.859043,-12.017938,-18.529509,-3.140957,0.0,0.002544,15.593722,4.484891,1.488443,20.789256,0.183456,0.183456,61.392979,92.580424,0.021021,0.200000,0.140765,0.243075,0.000000,0.200000,warning,high_lapse_estimate,256,8,200,parametric_binomial,4.605021,223.708875,psignifit_failed: 'dict' object has no attribu...,107.854438,0.025140,200,200,68.425702,8.919599,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,-12.017938,12.017938,12.017938,True,False,,3.982017,0.017511,0.244579
1,L_E_10,scipy_logistic_fallback,89.137498,4.516684,79.310970,96.877307,4.137498,-5.689030,11.877307,1.0,0.359642,20.865031,3.199417,12.760475,24.252867,0.245471,0.245471,69.035811,110.765874,0.013357,0.052460,0.000000,0.215762,0.000000,0.052460,ok,,256,8,200,parametric_binomial,18.025129,211.072580,psignifit_failed: 'dict' object has no attribu...,101.536290,0.059879,200,200,87.177755,17.680500,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,4.137498,-4.137498,4.137498,True,False,,4.516684,0.150123,0.285328
2,L_E_11,scipy_logistic_fallback,93.144585,6.128952,82.421825,107.402790,8.144585,-2.578175,22.402790,1.0,0.183891,44.728077,7.664089,24.568362,56.075815,0.526213,0.526213,59.903170,149.359323,0.007329,0.200000,0.000000,0.359510,0.000000,0.200000,warning,high_lapse_estimate,256,8,200,parametric_binomial,6.266557,292.104509,psignifit_failed: 'dict' object has no attribu...,142.052254,0.084326,200,200,80.075402,25.584430,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,8.144585,-8.144585,8.144585,True,False,,6.128952,0.289040,0.659715
3,L_E_12,scipy_logistic_fallback,84.378494,4.755407,73.205238,91.726926,-0.621506,-11.794762,6.726926,1.0,0.896017,15.617595,3.278818,6.860610,19.494511,0.183736,0.183736,69.801969,101.037159,0.018107,0.086223,0.007629,0.183250,0.000000,0.086223,ok,,256,8,200,parametric_binomial,7.162937,180.514350,psignifit_failed: 'dict' object has no attribu...,86.257175,0.042581,200,200,82.011760,12.503815,85.0,25.0,145.0,L_E,lab_experiment,airsled,with_airsled_L,NaN,other,unknown_success,L,60.0,60.0,L workspace (60x60 cm),experiment,unknown_sex,NaN,unknown_age,-0.621506,0.621506,0.621506,True,False,,4.755407,0.080713,0.229347
4,L_E_14,scipy_logistic_fallback,78.513260,6.646040,66.875849,91.400963,-6.486740,-18.124151,6.400963,1.0,0.329049,36.952520,8.404300,12.987496,46.727470,0.434736,0.434736,51.050553,124.955594,0.008871,0.200000,0.028073,0.392201,0.000000,0.200000,warning,high_lapse_estimate,256,8,200,parametric_binomial,6.538409,294.326289,psignifit_failed: 'dict' object has no attribu...,143.163144,0.079

## 10. Group-level fits by finger

Pools all valid trials across subjects within each finger condition.

In [268]:
# Fitted above in one shared parallel pool; display the finger-pooled result.
display(pse_jnd_group_by_finger)

,finger_condition,fit_method,pse,pse_se,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,standard_inside_pse_ci95,pse_bias_p_value,jnd,jnd_se,jnd_ci95_lower,jnd_ci95_upper,weber_fraction,jnd_over_standard,x25,x75,slope_at_pse,lapse_rate,lapse_rate_ci95_lower,lapse_rate_ci95_upper,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,n_bootstrap,bootstrap_method,deviance,aic,psignifit_status,neg_log_likelihood,lapse_rate_se,n_bootstrap_effective_pse,n_bootstrap_effective_jnd,mu,scale,standard_value,comparison_min,comparison_max,experiment_group,experiment_group_label,setup_factor,setup_label,protocol_group,protocol_group_label,success_label,workspace_setup,workspace_width_cm,workspace_height_cm,workspace_label,protocol_factor,sex_factor,age_years,age_group,pse_delta_comparison_minus_standard,pse_delta_standard_minus_comparison,abs_pse_delta_from_standard,pse_in_valid_band,excluded_from_group_analysis,group_exclusion_reason,pse_delta_se,jnd_over_standard_ci95_lower,jnd_over_standard_ci95_upper
0,I,scipy_logistic_fallback,85.545648,1.888568,81.725831,88.743946,0.545648,-3.274169,3.743946,1.0,0.772641,26.867196,1.977367,22.835112,30.582390,0.316085,0.316085,63.545755,117.280148,0.011407,0.176124,0.122296,0.234393,0.005654,0.170470,warning,high_lapse_estimate,2176,8,200,parametric_binomial,5.023916,2061.393868,psignifit_failed: 'dict' object has no attribu...,1026.696934,0.028816,200,200,78.515691,17.333651,85.0,25.0,145.0,NaN,other,NaN,unknown_setup,NaN,other,unknown_success,NaN,NaN,NaN,unknown workspace,unknown_protocol,unknown_sex,NaN,unknown_age,0.545648,-0.545648,0.545648,True,False,,1.888568,0.268648,0.359793
1,M,scipy_logistic_fallback,83.829492,2.139011,79.670388,87.697760,-1.170508,-5.329612,2.697760,1.0,0.584228,27.150480,1.991699,22.981247,31.080661,0.319417,0.319417,62.271612,116.572573,0.011661,0.209426,0.139043,0.260022,0.022921,0.186505,warning,high_lapse_estimate,2176,8,200,parametric_binomial,1.538376,2112.783601,psignifit_failed: 'dict' object has no attribu...,1052.391800,0.029674,200,200,77.017003,16.224068,85.0,25.0,145.0,NaN,other,NaN,unknown_setup,NaN,other,unknown_success,NaN,NaN,NaN,unknown workspace,unknown_protocol,unknown_sex,NaN,unknown_age,-1.170508,1.170508,1.170508,True,False,,2.139011,0.270368,0.365655
2,P,scipy_logistic_fallback,82.297059,2.056042,78.162329,86.212346,-2.702941,-6.837671,1.212346,1.0,0.188633,23.737386,2.017728,19.280489,28.108161,0.279263,0.279263,63.056216,110.530988,0.013095,0.195255,0.147047,0.244599,0.017389,0.177866,warning,high_lapse_estimate,2176,8,200,parametric_binomial,1.724874,2014.526651,psignifit_failed: 'dict' object has no attribu...,1003.263325,0.026162,200,200,76.333460,14.752438,85.0,25.0,145.0,NaN,other,NaN,unknown_setup,NaN,other,unknown_success,NaN,NaN,NaN,unknown workspace,unknown_protocol,unknown_sex,NaN,unknown_age,-2.702941,2.702941,2.702941,True,False,,2.056042,0.226829,0.330684
3,R,scipy_logistic_fallback,84.436397,1.797666,80.775112,87.938851,-0.563603,-4.224888,2.938851,1.0,0.753886,24.627884,2.069060,20.281313,28.525376,0.289740,0.289740,65.308035,114.563803,0.013050,0.214098,0.152703,0.244199,0.021293,0.192804,warning,high_lapse_estimate,2176,8,200,parametric_binomial,2.091765,2014.959269,psignifit_failed: 'dict' object has no attribu...,1003.479634,0.023356,200,200,78.075894,14.338251,85.0,25.0,145.0,NaN,other,NaN,unknown_setup,NaN,other,unknown_success,NaN,NaN,NaN,unknown workspace,unknown_protocol,unknown_sex,NaN,unknown_age,-0.563603,0.563603,0.563603,True,False,,1.797666,0.238604,0.335593


## 11. Group-level all-pooled fit

In [269]:
# Fitted above in one shared parallel pool; display the all-pooled result.
display(pse_jnd_group_all_pooled)

,group,fit_method,pse,pse_se,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,standard_inside_pse_ci95,pse_bias_p_value,jnd,jnd_se,jnd_ci95_lower,jnd_ci95_upper,weber_fraction,jnd_over_standard,x25,x75,slope_at_pse,lapse_rate,lapse_rate_ci95_lower,lapse_rate_ci95_upper,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,n_bootstrap,bootstrap_method,deviance,aic,psignifit_status,neg_log_likelihood,lapse_rate_se,n_bootstrap_effective_pse,n_bootstrap_effective_jnd,mu,scale,standard_value,comparison_min,comparison_max,experiment_group,experiment_group_label,setup_factor,setup_label,protocol_group,protocol_group_label,success_label,workspace_setup,workspace_width_cm,workspace_height_cm,workspace_label,protocol_factor,sex_factor,age_years,age_group,pse_delta_comparison_minus_standard,pse_delta_standard_minus_comparison,abs_pse_delta_from_standard,pse_in_valid_band,excluded_from_group_analysis,group_exclusion_reason,pse_delta_se,jnd_over_standard_ci95_lower,jnd_over_standard_ci95_upper
0,all_pooled,scipy_logistic_fallback,83.977923,1.064203,81.961832,86.128506,-1.022077,-3.038168,1.128506,1.0,0.336846,25.589488,1.012319,23.805835,27.774275,0.301053,0.301053,63.544226,114.723201,0.012277,0.200627,0.166311,0.229512,0.017461,0.183166,warning,high_lapse_estimate,8704,8,200,parametric_binomial,0.911094,8185.540095,psignifit_failed: 'dict' object has no attribu...,4088.770048,0.017019,200,200,77.424462,15.578125,85.0,25.0,145.0,NaN,other,NaN,unknown_setup,NaN,other,unknown_success,NaN,NaN,NaN,unknown workspace,unknown_protocol,unknown_sex,NaN,unknown_age,-1.022077,1.022077,1.022077,True,False,,1.064203,0.280069,0.326756


## 12. Subject-average group analysis

Computes subject-level probabilities first, then averages subjects so each subject has equal weight. This complements trial-pooled group fits.

In [270]:
subject_average_group_by_finger = pf.subject_average_psychometric(combined_clean_trials, ["finger_condition"])
subject_average_group_all = pf.subject_average_psychometric(combined_clean_trials.assign(group="all_subject_average"), ["group"])
pf.save_csv(subject_average_group_by_finger, OUTPUT_ROOT, "subject_average_group_by_finger.csv")
pf.save_csv(subject_average_group_all, OUTPUT_ROOT, "subject_average_group_all_pooled.csv")
display(subject_average_group_by_finger.head(30))

,finger_condition,comparison_value,n_subjects,n_data_points,mean_p_comparison_greater,mean_p_comparison_less,median_p_comparison_greater,median_p_comparison_less,sem_p_comparison_greater,sem_p_comparison_less,p_comparison_greater_ci95_lower,p_comparison_greater_ci95_upper,p_comparison_less_ci95_lower,p_comparison_less_ci95_upper,total_trials,median_rt,mean_rt,rt_ci95_lower,rt_ci95_upper,mean_log_reaction_time,log_reaction_time_ci95_lower,log_reaction_time_ci95_upper,standard_value,comparison_over_standard,signed_delta_over_standard,abs_delta_over_standard,geomean_reaction_time_s,reaction_time_log_ci95_lower_backtransformed_s,reaction_time_log_ci95_upper_backtransformed_s,delta_comparison_minus_standard,delta_standard_minus_comparison
0,I,25.0,36,288,0.076389,0.923611,0.0000,1.0000,0.028294,0.028294,0.020933,0.131845,0.868155,0.979067,288,1.946660,2.073341,1.978455,2.168227,0.708554,0.668401,0.748707,85.0,0.294118,-0.705882,0.705882,2.031052,1.951115,2.114264,-60.0,60.0
1,I,40.0,36,288,0.118056,0.881944,0.0000,1.0000,0.032254,0.032254,0.054839,0.181272,0.818728,0.945161,288,1.954245,2.040007,1.959474,2.120539,0.698069,0.662889,0.733249,85.0,0.470588,-0.529412,0.529412,2.009868,1.940390,2.081833,-45.0,45.0
2,I,55.0,36,288,0.187500,0.812500,0.1250,0.8750,0.035739,0.035739,0.117451,0.257549,0.742451,0.882549,288,1.959736,2.150625,2.012711,2.288540,0.728234,0.680530,0.775937,85.0,0.647059,-0.352941,0.352941,2.071419,1.974924,2.172628,-30.0,30.0
3,I,70.0,36,288,0.357639,0.642361,0.3750,0.6250,0.032334,0.032334,0.294265,0.421013,0.578987,0.705735,288,1.963639,2.199322,2.071607,2.327038,0.750546,0.702592,0.798499,85.0,0.823529,-0.176471,0.176471,2.118155,2.018980,2.222202,-15.0,15.0
4,I,100.0,36,288,0.642361,0.357639,0.6250,0.3750,0.034197,0.034197,0.575334,0.709388,0.290612,0.424666,288,1.955677,2.121517,2.021378,2.221655,0.728053,0.689896,0.766209,85.0,1.176471,0.176471,0.176471,2.071043,1.993508,2.151594,15.0,-15.0
5,I,115.0,36,288,0.684028,0.315972,0.7500,0.2500,0.033955,0.033955,0.617476,0.750579,0.249421,0.382524,288,1.983519,2.165292,2.062915,2.267669,0.739234,0.702932,0.775536,85.0,1.352941,0.352941,0.352941,2.094331,2.019665,2.171756,30.0,-30.0
6,I,130.0,36,288,0.802083,0.197917,0.8125,0.1875,0.026528,0.026528,0.750088,0.854079,0.145921,0.249912,288,1.972429,2.098901,2.009948,2.187854,0.722331,0.685213,0.759448,85.0,1.529412,0.529412,0.529412,2.059227,1.984194,2.137097,45.0,-45.0
7,I,145.0,36,288,0.784722,0.215278,0.8750,0.1250,0.033263,0.033263,0.719527,0.849918,0.150082,0.280473,288,1.914365,2.096609,1.991419,2.201799,0.712463,0.674605,0.750321,85.0,1.705882,0.705882,0.705882,2.039007,1.963257,2.117680,60.0,-60.0
8,M,25.0,36,288,0.097222,0.902778,0.0000,1.0000,0.033818,0.033818,0.030940,0.163505,0.836495,0.969060,288,1.964664,2.042461,1.958751,2.126170,0.697859,0.660953,0.734764,85.0,0.294118,-0.705882,0.705882,2.009445,1.936638,2.084990,-60.0,60.0
9,M,40.0,36,288,0.135417,0.864583,0.0000,1.0000,0.037059,0.037059,0.062781,0.208053,0.791947,0.937219,288,1.975033,2.058867,1.983599,2.134135,0.705748,0.670446,0.741049,85.0,0.470588,-0.529412,0.529412,2.025361,1.955110,2.098136,-45.0,45.0


## 11b. PSE uncertainty and bias sanity check

Each call to `pf.fit_conditions` now also runs a parametric binomial bootstrap (default `n_bootstrap=200`) to estimate, for every fit:

- `pse_se`, `pse_ci95_lower`, `pse_ci95_upper` - standard error and 95% percentile CI of the PSE on the original stimulus axis;
- `pse_delta_ci95_lower`, `pse_delta_ci95_upper` - the same CI expressed as comparison - standard so it can be compared against `0`;
- `standard_inside_pse_ci95` - boolean flag (`True` if the standard value falls within the 95% CI of the PSE, i.e. **no statistically significant bias**);
- `pse_bias_p_value` - Wald p-value for the null hypothesis `PSE == standard`;
- `jnd_se`, `jnd_ci95_lower`, `jnd_ci95_upper` and matching `lapse_rate_*` columns.

When `psignifit` is installed, the same columns are filled with its built-in Bayesian credible intervals instead (column `bootstrap_method == "psignifit_bayesian_ci"`).

The cells below print compact "bias verdict" tables so you can quickly spot fits where the PSE is significantly biased from the standard.

In [271]:
pse_bias_by_subject_finger = pf.pse_bias_summary(pse_jnd_by_subject_finger, ["subject_id", "finger_condition"])
pse_bias_subject_pooled = pf.pse_bias_summary(pse_jnd_subject_pooled, ["subject_id"])
pse_bias_group_by_finger = pf.pse_bias_summary(pse_jnd_group_by_finger, ["finger_condition"])
pse_bias_group_all_pooled = pf.pse_bias_summary(pse_jnd_group_all_pooled, ["group"])

pf.save_csv(pse_bias_by_subject_finger, OUTPUT_ROOT, "pse_bias_by_subject_finger.csv")
pf.save_csv(pse_bias_subject_pooled, OUTPUT_ROOT, "pse_bias_subject_pooled.csv")
pf.save_csv(pse_bias_group_by_finger, OUTPUT_ROOT, "pse_bias_group_by_finger.csv")
pf.save_csv(pse_bias_group_all_pooled, OUTPUT_ROOT, "pse_bias_group_all_pooled.csv")

print("=== Group-level fits (all subjects pooled, by finger) ===")
display(pse_bias_group_by_finger)
print("=== Group-level fit (all subjects, all fingers pooled) ===")
display(pse_bias_group_all_pooled)
print("=== Subject-pooled fits ===")
display(pse_bias_subject_pooled)

if not pse_bias_by_subject_finger.empty and "standard_inside_pse_ci95" in pse_bias_by_subject_finger.columns:
    flag = pse_bias_by_subject_finger["standard_inside_pse_ci95"]
    inside_mask = flag.astype("object").map(lambda v: bool(v) if pd.notna(v) else False)
    biased = pse_bias_by_subject_finger.loc[~inside_mask & flag.notna()]
    n_total = int(flag.notna().sum())
    n_biased = int((~inside_mask & flag.notna()).sum())
    print(
        f"\nPer-subject/finger fits with PSE significantly biased from the standard at 95% CI: "
        f"{n_biased} / {n_total} ({(n_biased / n_total * 100) if n_total else 0:.1f}%)."
    )
    if not biased.empty:
        print("Biased (subject, finger) fits:")
        display(
            biased[[
                "subject_id",
                "finger_condition",
                "pse",
                "pse_ci95_lower",
                "pse_ci95_upper",
                "pse_delta_from_standard",
                "pse_delta_ci95_lower",
                "pse_delta_ci95_upper",
                "pse_bias_p_value",
                "bias_verdict",
            ]]
        )

=== Group-level fits (all subjects pooled, by finger) ===


,finger_condition,n_trials,n_stimulus_levels,fit_method,standard_value,pse,pse_se,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,standard_inside_pse_ci95,pse_bias_p_value,jnd,jnd_ci95_lower,jnd_ci95_upper,lapse_rate,fit_warning,bias_verdict
0,I,2176,8,scipy_logistic_fallback,85.0,85.545648,1.888568,81.725831,88.743946,0.545648,-3.274169,3.743946,1.0,0.772641,26.867196,22.835112,30.582390,0.176124,high_lapse_estimate,standard within 95% CI (no significant bias)
1,M,2176,8,scipy_logistic_fallback,85.0,83.829492,2.139011,79.670388,87.697760,-1.170508,-5.329612,2.697760,1.0,0.584228,27.150480,22.981247,31.080661,0.209426,high_lapse_estimate,standard within 95% CI (no significant bias)
2,P,2176,8,scipy_logistic_fallback,85.0,82.297059,2.056042,78.162329,86.212346,-2.702941,-6.837671,1.212346,1.0,0.188633,23.737386,19.280489,28.108161,0.195255,high_lapse_estimate,standard within 95% CI (no significant bias)
3,R,2176,8,scipy_logistic_fallback,85.0,84.436397,1.797666,80.775112,87.938851,-0.563603,-4.224888,2.938851,1.0,0.753886,24.627884,20.281313,28.525376,0.214098,high_lapse_estimate,standard within 95% CI (no significant bias)


=== Group-level fit (all subjects, all fingers pooled) ===


,group,n_trials,n_stimulus_levels,fit_method,standard_value,pse,pse_se,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,standard_inside_pse_ci95,pse_bias_p_value,jnd,jnd_ci95_lower,jnd_ci95_upper,lapse_rate,fit_warning,bias_verdict
0,all_pooled,8704,8,scipy_logistic_fallback,85.0,83.977923,1.064203,81.961832,86.128506,-1.022077,-3.038168,1.128506,1.0,0.336846,25.589488,23.805835,27.774275,0.200627,high_lapse_estimate,standard within 95% CI (no significant bias)


=== Subject-pooled fits ===


,subject_id,n_trials,n_stimulus_levels,fit_method,standard_value,pse,pse_se,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,standard_inside_pse_ci95,pse_bias_p_value,jnd,jnd_ci95_lower,jnd_ci95_upper,lapse_rate,fit_warning,bias_verdict
0,L_E_1,256,8,scipy_logistic_fallback,85.0,72.982062,3.982017,66.470491,81.859043,-12.017938,-18.529509,-3.140957,0.0,0.002544,15.593722,1.488443,20.789256,0.200000,high_lapse_estimate,standard OUTSIDE 95% CI (significant bias)
1,L_E_10,256,8,scipy_logistic_fallback,85.0,89.137498,4.516684,79.310970,96.877307,4.137498,-5.689030,11.877307,1.0,0.359642,20.865031,12.760475,24.252867,0.052460,,standard within 95% CI (no significant bias)
2,L_E_11,256,8,scipy_logistic_fallback,85.0,93.144585,6.128952,82.421825,107.402790,8.144585,-2.578175,22.402790,1.0,0.183891,44.728077,24.568362,56.075815,0.200000,high_lapse_estimate,standard within 95% CI (no significant bias)
3,L_E_12,256,8,scipy_logistic_fallback,85.0,84.378494,4.755407,73.205238,91.726926,-0.621506,-11.794762,6.726926,1.0,0.896017,15.617595,6.860610,19.494511,0.086223,,standard within 95% CI (no significant bias)
4,L_E_14,256,8,scipy_logistic_fallback,85.0,78.513260,6.646040,66.875849,91.400963,-6.486740,-18.124151,6.400963,1.0,0.329049,36.952520,12.987496,46.727470,0.200000,high_lapse_estimate,standard within 95% CI (no significant bias)
5,L_E_15,256,8,scipy_logistic_fallback,85.0,92.533620,5.786759,81.797460,105.301645,7.533620,-3.202540,20.301645,1.0,0.192960,40.287282,22.364624,48.162853,0.200000,high_lapse_estimate,standard within 95% CI (no significant bias)
6,L_E_16,256,8,scipy_logistic_fallback,85.0,101.242395,8.825581,88.329777,121.393207,16.242395,3.329777,36.393207,0.0,0.065713,78.733793,42.213402,107.263524,0.200000,high_lapse_estimate,standard OUTSIDE 95% CI (significant bias)
7,L_E_17,256,8,scipy_logistic_fallback,85.0,91.730319,5.032450,81.138653,100.779185,6.730319,-3.861347,15.779185,1.0,0.181097,23.551087,12.945636,27.725152,0.200000,high_lapse_estimate,standard within 95% CI (no significant bias)
8,L_E_18,256,8,scipy_logistic_fallback,85.0,84.178822,6.428939,73.312391,97.104797,-0.821178,-11.687609,12.104797,1.0,0.898361,41.951610,23.416677,51.614347,0.200000,high_lapse_estimate,standard within 95% CI (no significant bias)
9,L_E_19,64,8,scipy_logistic_fallback,85.0,83.684395,209.123553,-579.351498,490.438031,-1.315605,-664.351498,405.438031,1.0,0.994981,230.257894,61.048225,2730.244573,0.227723,high_lapse_estimate,standard within 95% CI (no significant bias)



Per-subject/finger fits with PSE significantly biased from the standard at 95% CI: 32 / 144 (22.2%).
Biased (subject, finger) fits:


,subject_id,finger_condition,pse,pse_ci95_lower,pse_ci95_upper,pse_delta_from_standard,pse_delta_ci95_lower,pse_delta_ci95_upper,pse_bias_p_value,bias_verdict
2,L_E_1,P,69.531903,56.320824,84.286972,-15.468097,-28.679176,-0.713028,3.798558e-03,standard OUTSIDE 95% CI (significant bias)
5,L_E_10,M,69.132538,62.547376,70.008180,-15.867462,-22.452624,-14.991820,2.233103e-12,standard OUTSIDE 95% CI (significant bias)
18,L_E_14,P,54.377357,47.660032,62.057974,-30.622643,-37.339968,-22.942026,0.000000e+00,standard OUTSIDE 95% CI (significant bias)
22,L_E_15,P,111.976933,88.646895,144.376045,26.976933,3.646895,59.376045,5.338133e-02,standard OUTSIDE 95% CI (significant bias)
23,L_E_15,R,100.170906,86.445516,113.037874,15.170906,1.445516,28.037874,1.614204e-02,standard OUTSIDE 95% CI (significant bias)
29,L_E_17,M,112.808704,96.186042,131.470452,27.808704,11.186042,46.470452,1.616364e-03,standard OUTSIDE 95% CI (significant bias)
39,L_E_19,R,877.990749,109.585898,855.646038,792.990749,24.585898,770.646038,1.339475e-03,standard OUTSIDE 95% CI (significant bias)
41,L_E_2,M,85.076650,85.030736,98.065083,0.076650,0.030736,13.065083,9.871199e-01,standard OUTSIDE 95% CI (significant bias)
42,L_E_2,P,71.244401,70.354542,82.565304,-13.755599,-14.645458,-2.434696,6.436834e-03,standard OUTSIDE 95% CI (significant bias)
47,L_E_3,R,69.999997,68.587330,71.615346,-15.000003,-16.412670,-13.384654,1.998401e-15,standard OUTSIDE 95% CI (significant bias)


## 13. Order-effect analysis

Summarizes fatigue/order effects across trial number and inferred block order.

In [272]:
order_effects_summary, order_effects_binned = pf.compute_order_effects(combined_clean_trials)
pf.save_csv(order_effects_summary, OUTPUT_ROOT, "order_effects_summary.csv")
pf.save_csv(order_effects_binned, OUTPUT_ROOT, "order_effects_binned.csv")
display(order_effects_summary)

,subject_id,finger_condition,n_trials,n_data_points,first_half_p_comparison_greater,second_half_p_comparison_greater,first_half_mean_rt,second_half_mean_rt,median_rt,rt_ci95_lower,rt_ci95_upper,mean_log_reaction_time,median_log_reaction_time,log_reaction_time_ci95_lower,log_reaction_time_ci95_upper,spearman_response_vs_order,spearman_rt_vs_order,first_block_number,geomean_reaction_time_s,reaction_time_log_ci95_lower_backtransformed_s,reaction_time_log_ci95_upper_backtransformed_s
0,L_E_1,I,64,64,0.46875,0.43750,1.807114,1.777725,1.779743,1.766910,1.817928,0.582000,0.576469,0.568376,0.595625,-0.072214,-0.123581,4,1.789615,1.765397,1.814165
1,L_E_1,M,64,64,0.46875,0.56250,1.866421,1.823345,1.810256,1.812035,1.877731,0.610226,0.593468,0.594533,0.625919,0.085471,-0.249771,3,1.840848,1.812185,1.869964
2,L_E_1,P,64,64,0.40625,0.53125,1.906837,1.880968,1.869511,1.866466,1.921339,0.637018,0.625677,0.623149,0.650886,0.167803,-0.354075,2,1.890833,1.864791,1.917239
3,L_E_1,R,64,64,0.53125,0.34375,2.201796,2.065394,2.095478,2.082286,2.184905,0.753816,0.739781,0.732642,0.774990,-0.228475,-0.635302,1,2.125094,2.080570,2.170570
4,L_E_10,I,64,64,0.40625,0.43750,2.396017,1.963664,1.990096,2.004236,2.355445,0.750792,0.688182,0.699634,0.801951,-0.014558,-0.569093,1,2.118678,2.013015,2.229887
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,N_E_8,R,64,64,0.43750,0.56250,2.150770,1.970635,1.861604,1.910510,2.210895,0.694765,0.621438,0.640933,0.748597,0.143791,-0.350229,2,2.003238,1.898251,2.114032
140,N_E_9,I,64,64,0.50000,0.40625,2.896768,2.791573,2.399711,2.514839,3.173502,0.975815,0.875336,0.892580,1.059049,-0.058621,0.154670,1,2.653328,2.441421,2.883628
141,N_E_9,M,64,64,0.46875,0.46875,2.543844,2.398058,2.158315,2.154949,2.786953,0.849694,0.769238,0.781744,0.917645,0.015255,0.170330,2,2.338931,2.185279,2.503387
142,N_E_9,P,64,64,0.53125,0.43750,2.200259,2.314515,2.150948,2.156968,2.357806,0.800797,0.765908,0.762055,0.839539,-0.077008,0.060989,3,2.227316,2.142675,2.315300


## 14. Success-rate, answer-duration, and fatigue/learning analysis

This section tests the requested psychophysics-only questions:

- Does answer duration (`time_to_answer`) relate to success?
- Does success go up or down over the session (learning vs tiredness/fatigue proxy)?
- Are the effects visible within each participant/finger and across participants?

Because no subjective tiredness scale is present in `answers.csv`, tiredness is operationalized conservatively as trial order, elapsed time, and first-vs-second-half changes.


In [273]:
success_time_fatigue = pf.compute_success_time_fatigue(combined_success_trials, ["subject_id", "finger_condition"])

success_summary_by_subject_finger = success_time_fatigue["summary"]
success_trend_slopes_by_subject_finger = success_time_fatigue["slopes"]
success_by_reaction_time_bin = success_time_fatigue["reaction_time_bins"]
success_by_order_bin = success_time_fatigue["order_bins"]
fatigue_first_second_summary = success_time_fatigue["first_second"]
between_subject_success_time_stats = success_time_fatigue["between_subjects"]
success_summary_by_subject = success_time_fatigue["subject_summary"]

pf.save_csv(success_summary_by_subject, OUTPUT_ROOT, "success_summary_by_subject.csv")
pf.save_csv(success_summary_by_subject_finger, OUTPUT_ROOT, "success_summary_by_subject_finger.csv")
pf.save_csv(success_trend_slopes_by_subject_finger, OUTPUT_ROOT, "success_trend_slopes_by_subject_finger.csv")
pf.save_csv(success_by_reaction_time_bin, OUTPUT_ROOT, "success_by_reaction_time_bin.csv")
pf.save_csv(success_by_order_bin, OUTPUT_ROOT, "success_by_order_bin.csv")
pf.save_csv(fatigue_first_second_summary, OUTPUT_ROOT, "fatigue_first_second_summary.csv")
pf.save_csv(between_subject_success_time_stats, OUTPUT_ROOT, "between_subject_success_time_stats.csv")

print("Participant-level success summary")
display(success_summary_by_subject)
print("Within participant/finger success trend slopes: positive=success improves, negative=success decreases")
display(success_trend_slopes_by_subject_finger)
print("First vs second half fatigue/learning proxy")
display(fatigue_first_second_summary)
print("Between-participant duration/time statistics")
display(between_subject_success_time_stats)


Participant-level success summary


,subject_id,subject_group_label,n_trials,n_data_points,success_rate,success_rate_ci95_lower,success_rate_ci95_upper,mean_reaction_time,median_reaction_time,reaction_time_ci95_lower,reaction_time_ci95_upper,mean_log_reaction_time,log_reaction_time_ci95_lower,log_reaction_time_ci95_upper,session_duration_min,mean_abs_stiffness_delta,geomean_reaction_time_s,reaction_time_log_ci95_lower_backtransformed_s,reaction_time_log_ci95_upper_backtransformed_s
0,L_E_1,other,256,256,0.812500,0.760197,0.855563,1.916200,1.864899,1.892270,1.940130,0.645765,0.634338,0.657192,62.273045,37.5,1.907446,1.885773,1.929367
1,L_E_10,other,256,256,0.812500,0.760197,0.855563,1.991296,1.892819,1.933783,2.048809,0.673385,0.654461,0.692309,56.520598,37.5,1.960863,1.924105,1.998324
2,L_E_11,other,256,256,0.718750,0.660752,0.770280,2.351671,2.118362,2.256305,2.447037,0.818945,0.788782,0.849108,80.278058,37.5,2.268106,2.200714,2.337561
3,L_E_12,other,256,256,0.878906,0.833243,0.913365,2.066896,2.002182,2.029866,2.103926,0.718308,0.704066,0.732550,67.943181,37.5,2.050961,2.021957,2.080380
4,L_E_14,other,256,256,0.722656,0.664830,0.773899,2.076768,1.964511,1.993271,2.160265,0.711375,0.691555,0.731195,46.663964,37.5,2.036790,1.996819,2.077562
5,L_E_15,other,256,256,0.730469,0.673002,0.781121,2.471305,2.278803,2.390735,2.551875,0.881471,0.857261,0.905680,57.790664,37.5,2.414448,2.356698,2.473614
6,L_E_16,other,256,256,0.648438,0.588147,0.704339,2.153545,2.058420,2.100879,2.206212,0.752138,0.732201,0.772076,46.120512,37.5,2.121532,2.079654,2.164254
7,L_E_17,other,256,256,0.816406,0.764422,0.859035,2.107745,2.005069,2.059415,2.156075,0.733531,0.715920,0.751143,69.619694,37.5,2.082421,2.046068,2.119420
8,L_E_18,other,256,256,0.738281,0.681195,0.788322,2.463830,2.237280,2.356893,2.570766,0.870539,0.843602,0.897476,52.548685,37.5,2.388197,2.324726,2.453402
9,L_E_19,other,256,256,0.519531,0.458493,0.579992,2.134921,2.007748,2.083246,2.186597,0.743682,0.723916,0.763447,62.809944,37.5,2.103666,2.062495,2.145659


Within participant/finger success trend slopes: positive=success improves, negative=success decreases


,subject_id,finger_condition,n_trials,n_data_points,success_rate,success_rate_ci95_lower,success_rate_ci95_upper,median_reaction_time,reaction_time_ci95_lower,reaction_time_ci95_upper,mean_log_reaction_time,log_reaction_time_ci95_lower,log_reaction_time_ci95_upper,success_vs_order_slope,success_vs_order_p_value,success_vs_order_spearman_r,success_vs_order_spearman_p,success_vs_order_direction,success_vs_global_order_slope,success_vs_global_order_p_value,success_vs_global_order_spearman_r,success_vs_global_order_spearman_p,success_vs_global_order_direction,success_vs_elapsed_slope,success_vs_elapsed_p_value,success_vs_elapsed_direction,success_vs_reaction_time_slope,success_vs_reaction_time_p_value,success_vs_reaction_time_spearman_r,success_vs_reaction_time_spearman_p,success_vs_reaction_time_direction,geomean_reaction_time_s,reaction_time_log_ci95_lower_backtransformed_s,reaction_time_log_ci95_upper_backtransformed_s
0,L_E_1,I,64,64,0.890625,0.791011,0.946000,1.779743,1.766910,1.817928,0.582000,0.568376,0.595625,0.174519,0.195461,0.163958,0.195461,up_weak,0.739629,0.195461,0.163958,0.195461,up_weak,0.893841,0.170262,up_weak,-0.298911,0.436703,-0.177508,0.160543,down_weak,1.789615,1.765397,1.814165
1,L_E_1,M,64,64,0.859375,0.753835,0.924215,1.810256,1.812035,1.877731,0.610226,0.594533,0.625919,-0.108173,0.473347,-0.091241,0.473347,down_weak,-0.458448,0.473347,-0.091241,0.473347,down_weak,-0.566357,0.408463,down_weak,-0.027901,0.933270,-0.074210,0.560040,flat,1.840848,1.812185,1.869964
2,L_E_1,P,64,64,0.781250,0.665670,0.864978,1.869511,1.866466,1.921339,0.637018,0.623149,0.650886,0.103846,0.562972,0.073658,0.562972,up_weak,0.440110,0.562972,0.073658,0.562972,up_weak,0.182804,0.764557,up_weak,-1.016788,0.028896,-0.210742,0.094615,down_p_lt_0.05,1.890833,1.864791,1.917239
3,L_E_1,R,64,64,0.718750,0.598658,0.814068,2.095478,2.082286,2.184905,0.753816,0.732642,0.774990,-0.173077,0.374514,-0.112876,0.374514,down_weak,-0.733516,0.374514,-0.112876,0.374514,down_weak,-0.758355,0.356726,down_weak,0.443858,0.103946,0.244564,0.051459,up_weak,2.125094,2.080570,2.170570
4,L_E_10,I,64,64,0.765625,0.648665,0.852502,1.990096,2.004236,2.355445,0.750792,0.699634,0.801951,0.333173,0.066740,0.230623,0.066740,up_weak,1.412019,0.066740,0.230623,0.066740,up_weak,1.289883,0.046929,up_p_lt_0.05,-0.171919,0.020732,-0.156744,0.216126,down_p_lt_0.05,2.118678,2.013015,2.229887
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,N_E_8,R,64,64,0.843750,0.735717,0.912853,1.861604,1.910510,2.210895,0.694765,0.640933,0.748597,0.204808,0.191516,0.165396,0.191516,up_weak,0.867995,0.191516,0.165396,0.191516,up_weak,0.839968,0.177821,up_weak,-0.189311,0.010673,-0.223634,0.075670,down_p_lt_0.05,2.003238,1.898251,2.114032
140,N_E_9,I,64,64,0.828125,0.717866,0.901224,2.399711,2.514839,3.173502,0.975815,0.892580,1.059049,-0.246635,0.129168,-0.191688,0.129168,down_weak,-1.045261,0.129168,-0.191688,0.129168,down_weak,-0.558050,0.206645,down_weak,-0.132724,0.000092,-0.337416,0.006399,down_p_lt_0.05,2.653328,2.441421,2.883628
141,N_E_9,M,64,64,0.812500,0.700254,0.889355,2.158315,2.154949,2.786953,0.849694,0.781744,0.917645,-0.020192,0.905299,-0.015169,0.905299,flat,-0.085577,0.905299,-0.015169,0.905299,down_weak,-0.049091,0.955796,down_weak,-0.077834,0.041842,-0.153861,0.224795,down_p_lt_0.05,2.338931,2.185279,2.503387
142,N_E_9,P,64,64,0.765625,0.648665,0.852502,2.150948,2.156968,2.357806,0.800797,0.762055,0.839539,-0.073558,0.689477,-0.050917,0.689477,down_weak,-0.311745,0.689477,-0.050917,0.689477,down_weak,-0.461728,0.646805,down_weak,-0.305774,0.018565,-0.272554,0.029340,down_p_lt_0.05,2.227316,2.142675,2.315300


First vs second half fatigue/learning proxy


,subject_id,finger_condition,mean_reaction_time_first_half,mean_reaction_time_second_half,n_trials_first_half,n_trials_second_half,success_rate_first_half,success_rate_second_half,success_rate_second_minus_first,fatigue_direction,reaction_time_second_minus_first
0,L_E_1,I,1.807114,1.777725,32,32,0.84375,0.93750,0.09375,up_weak,-0.029389
1,L_E_1,M,1.866421,1.823345,32,32,0.90625,0.81250,-0.09375,down_weak,-0.043076
2,L_E_1,P,1.906837,1.880968,32,32,0.78125,0.78125,0.00000,flat,-0.025869
3,L_E_1,R,2.201796,2.065394,32,32,0.78125,0.65625,-0.12500,down_weak,-0.136402
4,L_E_10,I,2.396017,1.963664,32,32,0.65625,0.87500,0.21875,up_weak,-0.432353
...,...,...,...,...,...,...,...,...,...,...,...
139,N_E_8,R,2.150770,1.970635,32,32,0.81250,0.87500,0.06250,up_weak,-0.180135
140,N_E_9,I,2.896768,2.791573,32,32,0.87500,0.78125,-0.09375,down_weak,-0.105195
141,N_E_9,M,2.543844,2.398058,32,32,0.78125,0.84375,0.06250,up_weak,-0.145786
142,N_E_9,P,2.200259,2.314515,32,32,0.78125,0.75000,-0.03125,down_weak,0.114256


Between-participant duration/time statistics


,analysis,n_subjects,slope,linear_p_value,spearman_r,spearman_p,direction
0,between_subject_success_vs_mean_reaction_time,36,-0.353711,0.002023,-0.333891,0.046572,down_p_lt_0.05
1,between_subject_success_vs_median_reaction_time,36,-0.474881,0.004713,-0.364924,0.028643,down_p_lt_0.05
2,between_subject_success_vs_session_duration_min,36,0.000035,0.988179,0.142802,0.406062,flat
3,between_subject_success_vs_mean_abs_stiffness_...,36,NaN,NaN,NaN,NaN,unknown


## 15. Finger identity across time and finger appearance order

This section addresses two additional time questions:

1. **Between fingers across time:** compare index/middle/ring/pinky success trajectories across normalized time within each finger's trials.
2. **Appearance of fingers:** compare the first, second, third, and fourth finger encountered in each participant session, independent of the anatomical finger identity.

Outputs include subject-level slopes, group time-bin curves by finger, the finger-order matrix for every participant, and paired contrasts between fingers/appearance positions.


In [274]:
import importlib
pf = importlib.reload(pf)

finger_time_appearance = pf.compare_fingers_over_time_and_appearance(combined_success_trials)

finger_time_subject_summary = finger_time_appearance["subject_finger_summary"]
finger_time_group_bins = finger_time_appearance["group_finger_time_bins"]
finger_time_subject_bins = finger_time_appearance["subject_finger_time_bins"]
finger_appearance_order_summary = finger_time_appearance["appearance_order_summary"]
finger_by_appearance_order = finger_time_appearance["finger_by_appearance_order"]
finger_order_matrix = finger_time_appearance["finger_order_matrix"]
finger_time_slope_summary = finger_time_appearance["finger_slope_summary"]
stiffness_time_slope_summary = finger_time_appearance.get("stiffness_slope_summary", pd.DataFrame())
finger_time_slope_contrasts = finger_time_appearance["finger_slope_contrasts"]
finger_appearance_order_contrasts = finger_time_appearance["appearance_order_contrasts"]

pf.save_csv(finger_time_subject_summary, OUTPUT_ROOT, "finger_time_subject_summary.csv")
pf.save_csv(finger_time_group_bins, OUTPUT_ROOT, "finger_time_group_bins.csv")
pf.save_csv(finger_time_subject_bins, OUTPUT_ROOT, "finger_time_subject_bins.csv")
pf.save_csv(finger_appearance_order_summary, OUTPUT_ROOT, "finger_appearance_order_summary.csv")
pf.save_csv(finger_by_appearance_order, OUTPUT_ROOT, "finger_by_appearance_order.csv")
pf.save_csv(finger_order_matrix, OUTPUT_ROOT, "finger_order_matrix.csv")
pf.save_csv(finger_time_slope_summary, OUTPUT_ROOT, "finger_time_slope_summary.csv")
pf.save_csv(stiffness_time_slope_summary, OUTPUT_ROOT, "stiffness_time_slope_summary.csv")
pf.save_csv(finger_time_slope_contrasts, OUTPUT_ROOT, "finger_time_slope_contrasts.csv")
pf.save_csv(finger_appearance_order_contrasts, OUTPUT_ROOT, "finger_appearance_order_contrasts.csv")

print("Finger order matrix: which finger appeared 1st/2nd/3rd/4th for each participant")
display(finger_order_matrix)
print("Group success by finger and normalized within-finger time bin")
display(finger_time_group_bins)
print("Finger appearance-order summary")
display(finger_appearance_order_summary)
print("Finger time-slope summary")
display(finger_time_slope_summary)
print("Success slope by stiffness")
display(stiffness_time_slope_summary)
print("Paired contrasts between finger time slopes")
display(finger_time_slope_contrasts)
print("Paired contrasts between appearance-order success rates")
display(finger_appearance_order_contrasts)


Finger order matrix: which finger appeared 1st/2nd/3rd/4th for each participant


,subject_id,appearance_1_finger,appearance_2_finger,appearance_3_finger,appearance_4_finger
0,L_E_1,R,P,M,I
1,L_E_10,I,M,P,R
2,L_E_11,P,I,R,M
3,L_E_12,R,P,M,I
4,L_E_14,M,R,I,P
5,L_E_15,I,M,P,R
6,L_E_16,P,I,R,M
7,L_E_17,R,P,M,I
8,L_E_18,M,R,I,P
9,L_E_19,R,P,M,I


Group success by finger and normalized within-finger time bin


,finger_condition,within_finger_time_bin,n_subjects,total_trials,n_data_points,mean_success_rate,median_success_rate,sem_success_rate,success_rate_ci95_lower,success_rate_ci95_upper,mean_reaction_time,median_reaction_time,sem_reaction_time,reaction_time_ci95_lower,reaction_time_ci95_upper,mean_log_reaction_time,log_reaction_time_ci95_lower,log_reaction_time_ci95_upper,mean_within_finger_order_fraction,geomean_reaction_time_s,reaction_time_log_ci95_lower_backtransformed_s,reaction_time_log_ci95_upper_backtransformed_s
0,I,1,36,288,288,0.756944,0.7500,0.037909,0.682642,0.831247,2.211760,2.049621,0.064187,2.085954,2.337567,0.761122,0.716398,0.805845,0.055556,2.140676,2.047047,2.238587
1,I,2,36,288,288,0.802083,0.8750,0.028771,0.745693,0.858474,2.142064,1.941752,0.061873,2.020792,2.263336,0.730808,0.687082,0.774534,0.182540,2.076758,1.987906,2.169581
2,I,3,36,288,288,0.760417,0.7500,0.032034,0.697630,0.823203,2.123574,1.968835,0.060283,2.005419,2.241728,0.727822,0.684837,0.770806,0.309524,2.070565,1.983448,2.161508
3,I,4,36,288,288,0.788194,0.8750,0.034076,0.721405,0.854984,2.161035,1.940917,0.063540,2.036497,2.285573,0.738390,0.688746,0.788034,0.436508,2.092564,1.991217,2.199068
4,I,5,36,288,288,0.756944,0.7500,0.034122,0.690066,0.823823,2.126990,1.979553,0.065312,1.998979,2.255002,0.722384,0.678700,0.766068,0.563492,2.059337,1.971313,2.151291
5,I,6,36,288,288,0.795139,0.8125,0.026889,0.742436,0.847842,2.071157,1.951467,0.053747,1.965814,2.176500,0.703498,0.663340,0.743655,0.690476,2.020808,1.941265,2.103610
6,I,7,36,288,288,0.760417,0.7500,0.030446,0.700742,0.820091,2.099730,1.927076,0.047155,2.007305,2.192154,0.719010,0.679249,0.758771,0.817460,2.052400,1.972396,2.135650
7,I,8,36,288,288,0.753472,0.8125,0.039681,0.675698,0.831246,2.009304,1.902979,0.033846,1.942967,2.075641,0.684450,0.654027,0.714873,0.944444,1.982680,1.923270,2.043926
8,M,1,36,288,288,0.763889,0.7500,0.036180,0.692977,0.834801,2.231659,2.006013,0.074407,2.085822,2.377496,0.763455,0.712934,0.813977,0.055556,2.145677,2.039968,2.256865
9,M,2,36,288,288,0.725694,0.7500,0.033341,0.660347,0.791042,2.224434,1.967821,0.124767,1.979890,2.468977,0.739904,0.692760,0.787048,0.182540,2.095734,1.999227,2.196901


Finger appearance-order summary


...

,comparison_value,n_subjects,n_trials,mean_success_rate,mean_success_slope,median_success_slope,sem_success_slope,success_slope_ci95_lower,success_slope_ci95_upper
0,25.0,36,1152,0.917535,0.011086,0.000000,0.022502,-0.033019,0.055191
1,40.0,36,1152,0.881944,0.006965,0.000000,0.040478,-0.072372,0.086302
2,55.0,36,1152,0.801215,0.022093,-0.000378,0.043741,-0.063638,0.107825
3,70.0,36,1152,0.660590,-0.004993,0.003610,0.055242,-0.113267,0.103280
4,100.0,36,1152,0.653646,0.074521,0.101867,0.056327,-0.035880,0.184921
5,115.0,36,1152,0.723090,0.026209,0.032706,0.052603,-0.076892,0.129311
6,130.0,36,1152,0.765625,0.036298,0.057542,0.048245,-0.058263,0.130859
7,145.0,36,1152,0.782118,-0.001183,-0.031187,0.046568,-0.092457,0.090090


Paired contrasts between finger time slopes


,contrast,metric,n_paired_subjects,mean_difference,median_difference,sem_difference,difference_ci95_lower,difference_ci95_upper,t_stat,p_value
0,I_minus_M,success_vs_within_finger_time_slope,36,-0.018109,-0.027404,0.034695,-0.086111,0.049893,-0.521947,0.604995
1,I_minus_P,success_vs_within_finger_time_slope,36,-0.041186,-0.029567,0.036853,-0.113418,0.031046,-1.117569,0.271364
2,I_minus_R,success_vs_within_finger_time_slope,36,-0.031090,-0.051202,0.040629,-0.110723,0.048543,-0.765207,0.449278
3,M_minus_P,success_vs_within_finger_time_slope,36,-0.023077,-0.041106,0.034712,-0.091113,0.044959,-0.664810,0.510529
4,M_minus_R,success_vs_within_finger_time_slope,36,-0.012981,0.028846,0.040185,-0.091744,0.065782,-0.323025,0.748599
5,P_minus_R,success_vs_within_finger_time_slope,36,0.010096,-0.033173,0.035236,-0.058967,0.079159,0.286529,0.776162


Paired contrasts between appearance-order success rates


,contrast,metric,n_paired_subjects,mean_difference,median_difference,sem_difference,difference_ci95_lower,difference_ci95_upper,t_stat,p_value
0,appearance_1_minus_2,success_rate,36,-0.016927,-0.015625,0.009811,-0.036156,0.002302,-1.725355,0.093284
1,appearance_1_minus_3,success_rate,36,-0.027778,-0.007812,0.012153,-0.051597,-0.003958,-2.285714,0.028440
2,appearance_1_minus_4,success_rate,36,-0.013455,-0.023438,0.017343,-0.047447,0.020537,-0.775811,0.443072
3,appearance_2_minus_3,success_rate,36,-0.010851,0.000000,0.009760,-0.029981,0.008279,-1.111731,0.273832
4,appearance_2_minus_4,success_rate,36,0.003472,0.007812,0.012637,-0.021297,0.028241,0.274758,0.785117
5,appearance_3_minus_4,success_rate,36,0.014323,0.015625,0.013386,-0.011913,0.040559,1.070002,0.291942


## 16. Article-inspired figures adapted to this experiment: skin stretch and XY probing

The eLife article includes perception panels (psychometric curves, PSE, JND) and force/trajectory-style panels. For this experiment, the relevant substitutions are:

- **No grip force:** use the recorded skin-stretch gain/condition in **mm/m** and a skin-stretch command proxy, not grip force.
- **Probing from center in the XY plane:** use `object_x, object_y` from `tracking.csv` relative to screen center `(320, 240)`.
- **Article-style perception:** keep psychometric curves, PSE, JND, subject lines, and group average/dotted summary styling.

Important distinction: `motor_response_analisys_servo` is a separate hardware check/calibration for motor command response. This psychophysics notebook keeps the main analysis on the experiment variables that participants experienced: skin-stretch mm/m, XY probing from center, time, success, and finger/order effects.


In [275]:
article_style_figure_paths, article_style_figure_manifest = pf.save_article_style_psychophysics_figures(
    OUTPUT_ROOT,
    combined_success_trials,
    psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger,
    finger_time_subject_summary,
    finger_appearance_order_summary,
    finger_by_appearance_order,
    preferred_subject=None,
    fig_dpi=FIG_DPI,
)
print(f"Saved {len(article_style_figure_paths)} article-style psychophysics figures")
display(article_style_figure_manifest)


Saved 8 article-style psychophysics figures


,figure,selected_typical_subject,style_source
0,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted
1,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted
2,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted
3,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted
4,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted
5,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted
6,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted
7,c:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,,Farajian_Nisky_eLife_52653_Figure_7_adapted


## 17. Saved output tree

The final cell writes the requested architecture under `RUN_OUTPUT_ROOT`:

- group/all outputs under `results/<DATA_SELECTION>/`, for example `results/L_E/`
- subject outputs as sibling folders, for example `results/L_E_1/`, `results/L_E_2/`, `results/L_E_3/`, ...

Subject folders contain only subject-meaningful plots. The group folder contains one all-level plot set under figures/all with individual subjects faded in the background where relevant.

## Psychometric fit conventions, PSE band-pass, and figure encodings

**Symmetric delta axis.** All psychometric curves use the signed difference `G_comparison - G_standard` on the x-axis, centred on 0 (comparison == standard) and drawn symmetric over the full tested range (delta -80..+80, i.e. comparison 5..165 around the standard 85).

**PSE band-pass (LPF + HPF) for the group analysis.** A per-subject/finger fit is included in the *group* analysis only when its PSE lies inside the reliable stimulus band **25..145** (delta +/-60). Out-of-band or un-estimated PSE fits are kept and still drawn as individual curves, but are flagged (`pse_in_valid_band`, `excluded_from_group_analysis`, `group_exclusion_reason`) and **excluded from every group-level aggregation/comparison** (`compute_experiment_group_comparisons`). JND is not band-filtered.

**Unreliable CIs.** Degenerate fits (e.g. railed lapse rate, near-flat slope) can produce PSE/JND bootstrap CIs that run far outside the tested range. On the individual curves these CIs are **clipped to the +/-80 axis and labelled "CI unreliable"** so the figure stays readable; the raw values remain in the saved tables.

**Workspace shading (group per-finger curves).** Background individual curves are shaded by workspace: **L workspace in a darker shade, N workspace in a lighter shade** (see legend).

**Pilot exclusion.** Pilot/protocol subjects (`*_P`, i.e. L_P/N_P) are pilot data, not experiment results, so they are excluded from every group-level fit (subject-pooled, per-finger, all-pooled) AND from the group-curve backgrounds; together with the band-pass-disqualified subjects they never enter the group analysis. They still appear as individual per-subject curves.

In [ ]:
# Group comparisons for the active (pooled) selection; reused by the main tree.
psychophysics_group_comparisons = pf.save_experiment_group_comparison_outputs(
    OUTPUT_ROOT,
    combined_success_trials,
    pse_jnd_by_subject_finger=pse_jnd_by_subject_finger,
    pse_jnd_subject_pooled=pse_jnd_subject_pooled,
)
motor_control_method_references = pf.motor_control_method_references()
pf.save_csv(motor_control_method_references, OUTPUT_ROOT, "motor_control_method_references.csv")


def _filter_to_group(frame, code):
    """Rows whose subject_id belongs to experiment-group ``code`` (e.g. 'L_E')."""
    if frame is None:
        return frame
    if frame.empty or "subject_id" not in frame.columns:
        return frame.iloc[0:0].copy()
    keep = frame["subject_id"].astype(str).map(lambda s: pf.subject_group_code(s) == code)
    return frame[keep].copy()


def _save_analysis_tree(out_root, selection, clean_sub, success_sub, flagged_sub, qc_sub,
                        psy_by_sf, fits_by_sf, aggs, group_comparisons):
    return pf.save_selected_analysis_tree(
        out_root,
        selection,
        clean=clean_sub,
        flagged=flagged_sub,
        success_trials=success_sub,
        qc_summary=qc_sub,
        psychometric_input_by_subject_finger=psy_by_sf,
        pse_jnd_by_subject_finger=fits_by_sf,
        psychometric_input_group_by_finger=aggs["psychometric_input_group_by_finger"],
        pse_jnd_group_by_finger=aggs["pse_jnd_group_by_finger"],
        psychometric_input_group_all_pooled=aggs["psychometric_input_group_all_pooled"],
        pse_jnd_group_all_pooled=aggs["pse_jnd_group_all_pooled"],
        order_effects_summary=aggs["order_effects_summary"],
        order_effects_binned=aggs["order_effects_binned"],
        success_time_fatigue=aggs["success_time_fatigue"],
        finger_time_appearance=aggs["finger_time_appearance"],
        psychophysics_group_comparisons=group_comparisons,
        fig_dpi=FIG_DPI,
    )


# For the combined L_N_E cohort, also emit a full standalone group analysis for
# each sub-cohort (L_E, N_E) as sibling folders -- identical to running each on
# its own -- in addition to the pooled L_N_E tree written below.
_subgroup_trees = []
if pf.normalize_data_selection(DATA_SELECTION) == "L_N_E":
    for _code in pf.GROUP_SELECTIONS["L_N_E"]:  # ("L_E", "N_E")
        _clean_sub = _filter_to_group(combined_clean_trials, _code)
        if _clean_sub.empty:
            print(f"No {_code} subjects in this run; skipping {_code} tree.")
            continue
        _success_sub = _filter_to_group(combined_success_trials, _code)
        _flagged_sub = _filter_to_group(combined_flagged_trials, _code)
        _psy_sub = _filter_to_group(psychometric_input_by_subject_finger, _code)
        _fits_sub = _filter_to_group(pse_jnd_by_subject_finger, _code)
        _qc_sub = pf.make_qc_summary(_clean_sub, _flagged_sub)
        _aggs = pf.compute_psychophysics_group_aggregates(
            _clean_sub, _success_sub, _psy_sub, _fits_sub, psignifit_available=PSIGNIFIT_AVAILABLE
        )
        _work = OUTPUT_BASE / f"_working_{_code}"
        pf.reset_output_root(_work)
        _grp_cmp = pf.save_experiment_group_comparison_outputs(
            _work,
            _success_sub,
            pse_jnd_by_subject_finger=_fits_sub,
            pse_jnd_subject_pooled=_aggs["pse_jnd_subject_pooled"],
        )
        _out = OUTPUT_BASE / _code
        pf.reset_output_root(_out)
        _save_analysis_tree(_out, _code, _clean_sub, _success_sub, _flagged_sub, _qc_sub,
                            _psy_sub, _fits_sub, _aggs, _grp_cmp)
        try:
            pf._rmtree_windows_retry(_work)
        except OSError:
            pass
        _subgroup_trees.append(_out)
        print(f"Saved standalone sub-cohort tree: {_out}")

# Main (pooled) tree for the active selection. For L_N_E this is the combined
# L_E + N_E pooled analysis; for any other selection it is the only tree.
analysis_tree_manifest = pf.save_selected_analysis_tree(
    RUN_OUTPUT_ROOT,
    DATA_SELECTION,
    clean=combined_clean_trials,
    flagged=combined_flagged_trials,
    success_trials=combined_success_trials,
    qc_summary=qc_summary,
    psychometric_input_by_subject_finger=psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger=pse_jnd_by_subject_finger,
    psychometric_input_group_by_finger=psychometric_input_group_by_finger,
    pse_jnd_group_by_finger=pse_jnd_group_by_finger,
    psychometric_input_group_all_pooled=psychometric_input_group_all_pooled,
    pse_jnd_group_all_pooled=pse_jnd_group_all_pooled,
    order_effects_summary=order_effects_summary,
    order_effects_binned=order_effects_binned,
    success_time_fatigue=success_time_fatigue,
    finger_time_appearance=finger_time_appearance,
    psychophysics_group_comparisons=psychophysics_group_comparisons,
    fig_dpi=FIG_DPI,
    write_subjects=not bool(_subgroup_trees),
)

print(f"Saved requested output tree for {DATA_SELECTION}: {RUN_OUTPUT_ROOT}")
if _subgroup_trees:
    print("Sub-cohort trees:", _subgroup_trees)
print("Group figures:", RUN_OUTPUT_ROOT / "figures")
print("Group CSVs:", RUN_OUTPUT_ROOT / "csv")
print("Subject folders:", [OUTPUT_BASE / pf.sanitize_name(s) for s in selected_subject_ids])
print("Tree manifest:", RUN_OUTPUT_ROOT / "analysis_tree_manifest.csv")
display(analysis_tree_manifest.head(80))

print("N_E / L_E / L_P / N_P psychophysics group summary where available")
for key in [
    "psychophysics_trial_group_metric_summary",
    "psychophysics_protocol_group_metric_summary",
    "psychophysics_analysis_scope_metric_summary",
]:
    if key in psychophysics_group_comparisons and not psychophysics_group_comparisons[key].empty:
        print(key)
        display(psychophysics_group_comparisons[key].head(80))

print("Motor-control / psychophysics method references saved for later citation")
display(motor_control_method_references[["citation_key", "year", "title", "doi", "analysis_use"]])

working_manifest = pf.analysis_manifest(OUTPUT_ROOT)
display(working_manifest)

if not KEEP_WORKING_OUTPUTS and OUTPUT_ROOT.exists():
    # Robust removal: the intermediate scratch folder can hold a transient
    # Windows/Dropbox file lock. Retry, and never fail the run over scratch.
    try:
        pf._rmtree_windows_retry(OUTPUT_ROOT)
        print("Removed intermediate working folder:", OUTPUT_ROOT)
    except OSError as exc:
        print("Left intermediate working folder in place (locked):", OUTPUT_ROOT, "->", exc)


## Interpretation notes

- `PSE` is the physical comparison value where `P(response_comparison_greater) = 0.5`.
- `JND = (x75 - x25) / 2`, in the same units as the stimulus values.
- `farajian_style_input_by_subject_finger.csv` mirrors the Farajian/Nisky perception scripts: stimulus difference, number of comparison-stiffer responses, and total repetitions.
- `correct_response` is the success variable: 1 means the physically stiffer stimulus was selected, 0 means it was not.
- `success_vs_order_slope > 0` suggests improvement/learning over the session; `< 0` suggests possible fatigue or decline. Treat this as a proxy because no direct subjective tiredness rating is present in `answers.csv`.
- `success_vs_reaction_time_slope` and `success_by_reaction_time_bin.csv` show whether longer answer duration is associated with better or worse success.
- `finger_time_group_bins.csv` compares success across normalized time within each finger, enabling direct index/middle/ring/pinky time-course comparison.
- `finger_appearance_order_summary.csv` compares the 1st/2nd/3rd/4th finger encountered in a participant session, which tests whether finger appearance/order itself matters.
- `finger_order_matrix.csv` records which anatomical finger appeared in each order position for every participant.
- Review `fit_warning`, `qc_warnings`, `combined_flagged_trials.csv`, and the per-subject trend tables before interpreting results.
- `motor_response_analisys_servo` is treated as a separate hardware calibration/check, not mixed into the main psychophysics variables.
- Rows excluded from fitting are preserved for audit; no raw data files are modified.

- All psychophysics group outputs now include the requested scopes when available: protocol/group (`N_E`, `L_E`, `L_P`, broad `N/L/P/E`, and all participants), participant, finger, stiffness/comparison value, success/failure, sex, age group, and workspace (`N` 40x50 cm; `L` 60x60 cm).
- `comparison_over_standard`, `signed_delta_over_standard`, and `abs_delta_over_standard` normalize analog stiffness values to the trial standard while preserving original values; workspace columns document the original physical space and do not alter stimulus units.
- Scope-summary plots use faded raw/background values plus the summary bar/point so every value remains transparent behind the group statistic.
- `psychophysics_factor_statistics.csv` is an ANOVA-style screening table. For binary repeated 2AFC trials, treat it as descriptive evidence; for manuscript-level inference prefer planned psychometric model comparisons or mixed/repeated-measures models with participant as a repeated/random factor.
- `motor_control_method_references.csv` lists the motor-control and haptic psychophysics articles used to guide the analysis additions and citation trail.
